<a href="https://colab.research.google.com/github/belvinos91/pythonProject/blob/master/DuoIHM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Retry with lighter workloads and stepwise execution to avoid timeouts.
import numpy as np
import matplotlib.pyplot as plt
import os

OUTDIR = "/content/data/ihm_figs"
os.makedirs(OUTDIR, exist_ok=True)
rng = np.random.default_rng(7)

def sigma_eff_from_tuple(sigP, sigQ, rho):
    return float(np.sqrt(sigP**2 + sigQ**2 - 2*rho*sigP*sigQ))

def L_affine(p, A, B, sigP, sigQ, rho):
    return (A + B*p) + 2*(sigP**2*(1-p) - sigQ**2*p) + 2*rho*sigP*sigQ*(2*p - 1)

def euler_maruyama_paths(p0, T, dt, n_paths, A, B, sigP, sigQ, rho, clip_reflect=True):
    sigma_eff = sigma_eff_from_tuple(sigP, sigQ, rho)
    n_steps = int(np.round(T/dt))
    t = np.linspace(0.0, n_steps*dt, n_steps+1)
    P = np.empty((n_paths, n_steps+1), dtype=float)
    P[:,0] = p0
    for k in range(n_steps):
        p = P[:,k]
        Lval = L_affine(p, A, B, sigP, sigQ, rho)
        b = p*(1-p)*Lval
        z = rng.standard_normal(n_paths)
        incr = b*dt + sigma_eff*p*(1-p)*np.sqrt(dt)*z
        pk1 = p + incr
        if clip_reflect:
            too_low = pk1 <= 0.0
            pk1[too_low] = -pk1[too_low]
            too_high = pk1 >= 1.0
            pk1[too_high] = 2.0 - pk1[too_high]
            pk1 = np.clip(pk1, 1e-12, 1-1e-12)
        P[:,k+1] = pk1
    return t, P

def logit(p):
    return np.log(p/(1-p))

def mu_Y(p, A, B, sigP, sigQ, rho):
    Lval = L_affine(p, A, B, sigP, sigQ, rho)
    sigma_eff = sigma_eff_from_tuple(sigP, sigQ, rho)
    return Lval + 0.5*sigma_eff**2*(2*p - 1)

def pstar_affine(A, B, sigP, sigQ, rho):
    C0 = A + 2*sigP**2 - 2*rho*sigP*sigQ
    C1 = B - 2*sigP**2 - 2*sigQ**2 + 4*rho*sigP*sigQ
    if abs(C1) < 1e-12:
        return np.nan
    return -C0/C1

# --- Fig 1 (lighter) ---
def figure1():
    p_targets = [0.25, 0.5, 0.75]
    rhos = [-0.9, 0.0, 0.9]
    h_list = np.array([0.003, 0.006, 0.012, 0.024])
    n_paths = 3000
    dt = 2e-4
    A = 0.0; B = 0.0; sigP=0.2; sigQ=0.3
    plt.figure(figsize=(6.0,4.5))
    for p0 in p_targets:
        for rho in rhos:
            sig_eff = sigma_eff_from_tuple(sigP, sigQ, rho)
            vars_emp = []
            for h in h_list:
                t, P = euler_maruyama_paths(p0, float(h), dt, n_paths, A,B,sigP,sigQ,rho, True)
                dps = P[:,-1] - P[:,0]
                vars_emp.append(np.var(dps))
            th = (sig_eff**2)*(p0**2)*((1-p0)**2)*h_list
            plt.plot(h_list, vars_emp, marker='o', linestyle='None', label=f"p0={p0}, ρ={rho}")
            plt.plot(h_list, th, linestyle='-')
    plt.xlabel("h"); plt.ylabel("Var(Δp)"); plt.title("Variance scaling vs h")
    plt.legend(fontsize=8, ncol=2)
    p1 = os.path.join(OUTDIR, "fig1_variance_scaling.png")
    q1 = os.path.join(OUTDIR, "fig1_variance_scaling.pdf")
    plt.tight_layout(); plt.savefig(p1, dpi=200); plt.savefig(q1); plt.close()
    return p1, q1

# --- Fig 2 ---
def figure2():
    A = 0.1; B = 0.3; sigP=0.2; sigQ=0.3; rho=0.2
    T=3.0; dt=1e-3; p0=0.3
    t,P=euler_maruyama_paths(p0,T,dt,1,A,B,sigP,sigQ,rho)
    p=P[0]; Y=logit(p); dY=np.diff(Y); y=dY/dt; x=p[:-1]
    pgrid=np.linspace(0.05,0.95,200); theo=mu_Y(pgrid,A,B,sigP,sigQ,rho)
    plt.figure(figsize=(6.0,4.5))
    plt.plot(x[::20], y[::20], marker='o', linestyle='None', alpha=0.6, label="ΔY/Δt (scatter)")
    plt.plot(pgrid, theo, label="theory μ_Y(p)")
    plt.xlabel("p"); plt.ylabel("ΔY/Δt"); plt.title("Logit drift: data vs theory")
    plt.legend()
    p2 = os.path.join(OUTDIR, "fig2_logit_regression.png")
    q2 = os.path.join(OUTDIR, "fig2_logit_regression.pdf")
    plt.tight_layout(); plt.savefig(p2, dpi=200); plt.savefig(q2); plt.close()
    return p2, q2

# --- Fig 3 suite ---
def hist_stationary(A,B,sigP,sigQ,rho,fprefix):
    t,P=euler_maruyama_paths(0.3,150.0,1e-3,1,A,B,sigP,sigQ,rho)
    p=P[0]; p_ss=p[t>=40.0]; pstar=pstar_affine(A,B,sigP,sigQ,rho)
    plt.figure(figsize=(6.0,4.5))
    plt.hist(p_ss, bins=60, density=True)
    if np.isfinite(pstar) and 0<pstar<1:
        plt.axvline(pstar, linestyle='--', label=f"p* ≈ {pstar:.3f}")
    plt.xlabel("p"); plt.ylabel("density")
    plt.title(f"Stationary prevalence: A={A:.2f}, B={B:.2f}, ρ={rho:.2f}")
    if np.isfinite(pstar) and 0<pstar<1: plt.legend()
    pth = os.path.join(OUTDIR, f"{fprefix}.png")
    qth = os.path.join(OUTDIR, f"{fprefix}.pdf")
    plt.tight_layout(); plt.savefig(pth, dpi=200); plt.savefig(qth); plt.close()
    return pth, qth

def figure3():
    a1 = hist_stationary(-0.2, 0.6, 0.2, 0.3, 0.3, "fig3a_wolbachia")
    a2 = hist_stationary(0.05, 0.0, 0.2, 0.25, 0.0, "fig3b_resistance")
    a3 = hist_stationary(0.1, -0.4, 0.25, 0.2, 0.6, "fig3c_genedrive")
    return a1 + a2 + a3

# --- Fig 4 MB ---
def L_MB(p, rphi, tau, lam, beta, N, sigP, sigQ, rho):
    A_MB = rphi*(tau-1) + lam + beta*N
    B_MB = - rphi*(1-tau)
    return (A_MB + 2*(sigP**2*(1-p) - sigQ**2*p) + 2*rho*sigP*sigQ*(2*p-1)) + B_MB*(p/(1-p))

def figure4():
    rphi=1.0; tau=0.8; lam=0.0; beta=0.1; N=1.0; sigP=0.2; sigQ=0.25; rho=0.2
    pgrid=np.linspace(0.01,0.99,400); Lvals=L_MB(pgrid,rphi,tau,lam,beta,N,sigP,sigQ,rho)
    plt.figure(figsize=(6.0,4.5))
    plt.plot(pgrid, Lvals); plt.axhline(0.0, linestyle='--')
    plt.xlabel("p"); plt.ylabel("L_MB(p)"); plt.title("MB: L(p) monotone; unique zero if entry>0")
    p4 = os.path.join(OUTDIR, "fig4_mb_uniqueness.png")
    q4 = os.path.join(OUTDIR, "fig4_mb_uniqueness.pdf")
    plt.tight_layout(); plt.savefig(p4, dpi=200); plt.savefig(q4); plt.close()
    return p4, q4

# --- Fig 5 estimation (lighter) ---
def estimate_sigma_eff_methods(p0, A, B, sigP, sigQ, rho, T, dt):
    sigma_eff = sigma_eff_from_tuple(sigP, sigQ, rho)
    t,P=euler_maruyama_paths(p0,T,dt,1,A,B,sigP,sigQ,rho)
    p=P[0]
    dp=np.diff(p)
    Lvals=L_affine(p[:-1],A,B,sigP,sigQ,rho)
    b=p[:-1]*(1-p[:-1])*Lvals
    num=np.sum((dp - b*dt)**2); den=np.sum(p[:-1]**2*(1-p[:-1])**2*dt)
    sighat1 = num/den
    Y=logit(p); dY=np.diff(Y)
    mu = Lvals + 0.5*(sigma_eff**2)*(2*p[:-1]-1)
    sighat2 = (1/(len(dY)*dt))*np.sum((dY - mu*dt)**2)
    return sigma_eff, sighat1, sighat2

def figure5():
    p0=0.3; A=0.1; B=0.2; sigP=0.2; sigQ=0.3; rho=0.2
    dts = np.array([0.004, 0.002, 0.001])
    nrep = 25
    biases1=[]; rmses1=[]; biases2=[]; rmses2=[]
    true_sq = sigma_eff_from_tuple(sigP,sigQ,rho)**2
    for dt in dts:
        e1=[]; e2=[]
        for _ in range(nrep):
            _, h1, h2 = estimate_sigma_eff_methods(p0,A,B,sigP,sigQ,rho,T=3.0,dt=float(dt))
            e1.append(h1 - true_sq); e2.append(h2 - true_sq)
        e1=np.array(e1); e2=np.array(e2)
        biases1.append(np.mean(e1)); rmses1.append(np.sqrt(np.mean(e1**2)))
        biases2.append(np.mean(e2)); rmses2.append(np.sqrt(np.mean(e2**2)))
    plt.figure(figsize=(6.0,4.5))
    plt.plot(dts, biases1, marker='o', label="E1 bias (σ_eff^2)")
    plt.plot(dts, biases2, marker='s', label="E2 bias (σ_eff^2)")
    plt.gca().invert_xaxis()
    plt.xlabel("Δt"); plt.ylabel("Bias"); plt.title("Bias vs Δt (σ_eff^2 estimators)"); plt.legend()
    p5a=os.path.join(OUTDIR,"fig5a_estimation_bias.png"); q5a=os.path.join(OUTDIR,"fig5a_estimation_bias.pdf")
    plt.tight_layout(); plt.savefig(p5a, dpi=200); plt.savefig(q5a); plt.close()
    plt.figure(figsize=(6.0,4.5))
    plt.plot(dts, rmses1, marker='o', label="E1 RMSE (σ_eff^2)")
    plt.plot(dts, rmses2, marker='s', label="E2 RMSE (σ_eff^2)")
    plt.gca().invert_xaxis()
    plt.xlabel("Δt"); plt.ylabel("RMSE"); plt.title("RMSE vs Δt (σ_eff^2 estimators)"); plt.legend()
    p5b=os.path.join(OUTDIR,"fig5b_estimation_rmse.png"); q5b=os.path.join(OUTDIR,"fig5b_estimation_rmse.pdf")
    plt.tight_layout(); plt.savefig(p5b, dpi=200); plt.savefig(q5b); plt.close()
    return p5a, q5a, p5b, q5b

# --- Fig 6 ---
def figure6():
    sigma_eff_target = 0.25
    triplets = [(0.25, 0.0, 0.0), (0.3, 0.1, 0.0), (0.2, 0.2, -0.5)]
    adj = []
    for sigP, sigQ, rho in triplets:
        if sigP*sigQ > 0:
            rho_req = (sigP**2 + sigQ**2 - sigma_eff_target**2) / (2*sigP*sigQ)
            rho_req = float(np.clip(rho_req, -1.0, 1.0))
            adj.append((sigP,sigQ,rho_req))
        else:
            if sigP == 0.0:
                adj.append((0.0, sigma_eff_target, 0.0))
            else:
                adj.append((sigma_eff_target, 0.0, 0.0))
    A=0.05; B=0.1; T=4.0; dt=1e-3; p0=0.4
    plt.figure(figsize=(6.0,4.5))
    for i,(sigP,sigQ,rho) in enumerate(adj, start=1):
        t,P=euler_maruyama_paths(p0,T,dt,1,A,B,sigP,sigQ,rho)
        plt.plot(t, P[0], label=f"#{i}: σP={sigP:.2f}, σQ={sigQ:.2f}, ρ={rho:.2f}, σ_eff={sigma_eff_from_tuple(sigP,sigQ,rho):.2f}")
    plt.xlabel("time"); plt.ylabel("p(t)"); plt.title("Same σ_eff via different (σP,σQ,ρ)")
    plt.legend(fontsize=8)
    p6=os.path.join(OUTDIR,"fig6_same_sigmaeff_paths.png"); q6=os.path.join(OUTDIR,"fig6_same_sigmaeff_paths.pdf")
    plt.tight_layout(); plt.savefig(p6, dpi=200); plt.savefig(q6); plt.close()
    return p6, q6

# Execute stepwise
paths = []
paths += list(figure1())
paths += list(figure2())
paths += list(figure3())
paths += list(figure4())
paths += list(figure5())
paths += list(figure6())

paths


['/content/data/ihm_figs/fig1_variance_scaling.png',
 '/content/data/ihm_figs/fig1_variance_scaling.pdf',
 '/content/data/ihm_figs/fig2_logit_regression.png',
 '/content/data/ihm_figs/fig2_logit_regression.pdf',
 '/content/data/ihm_figs/fig3a_wolbachia.png',
 '/content/data/ihm_figs/fig3a_wolbachia.pdf',
 '/content/data/ihm_figs/fig3b_resistance.png',
 '/content/data/ihm_figs/fig3b_resistance.pdf',
 '/content/data/ihm_figs/fig3c_genedrive.png',
 '/content/data/ihm_figs/fig3c_genedrive.pdf',
 '/content/data/ihm_figs/fig4_mb_uniqueness.png',
 '/content/data/ihm_figs/fig4_mb_uniqueness.pdf',
 '/content/data/ihm_figs/fig5a_estimation_bias.png',
 '/content/data/ihm_figs/fig5a_estimation_bias.pdf',
 '/content/data/ihm_figs/fig5b_estimation_rmse.png',
 '/content/data/ihm_figs/fig5b_estimation_rmse.pdf',
 '/content/data/ihm_figs/fig6_same_sigmaeff_paths.png',
 '/content/data/ihm_figs/fig6_same_sigmaeff_paths.pdf']

In [ ]:
# IHM Malliavin derivative heatmaps: E[|D_s p(t)|] and Corr(D_s p(t), p(t))
# - Autonomous affine-contrast example (Wolbachia-like)
# - Euler–Maruyama simulation for prevalence p
# - Tangent SDE for Malliavin derivative Z_{s->t} obeys
#   dZ_t = ∂_p b(p_t) Z_t dt + ∂_p σ(p_t) Z_t dW_t,  with  Z_s = σ(p_s) = σ_eff p_s(1-p_s)
#
# Figures:
#   - fig_IHM_malliavin_heatmap_meanabs.png/pdf  (E[|D_s p(t)|])
#   - fig_IHM_malliavin_heatmap_corr.png/pdf     (Corr(D_s p(t), p(t)))
#
# This code uses only numpy/matplotlib; no seaborn; one chart per figure; default colors.

import numpy as np
import matplotlib.pyplot as plt
import os

# ---------------------- Parameters ----------------------
T = 5.0                     # time horizon
dt = 0.002                  # time step (smaller -> more accurate)
n_steps = int(T/dt)
times = np.linspace(0, T, n_steps+1)

# Prevalence SDE parameters (autonomous affine contrast Δr = A + B p)
A = -0.10                   # intercept (e.g., cost/baseline disadvantage)
B =  0.80                   # slope (e.g., CI strength)
sigma_P = 0.30
sigma_Q = 0.20
rho = 0.6

sigma_eff = np.sqrt(sigma_P**2 + sigma_Q**2 - 2*rho*sigma_P*sigma_Q)

# L(p) and b(p)
def L_of_p(p):
    # clamp p away from 0/1 to avoid numerical blow-ups, but return separate array
    p = np.clip(p, 1e-9, 1-1e-9)
    return (A + B*p) + 2*(sigma_P**2*(1-p) - sigma_Q**2*p) + 2*rho*sigma_P*sigma_Q*(2*p - 1)

def b_of_p(p):
    p = np.clip(p, 1e-9, 1-1e-9)
    return p*(1-p)*L_of_p(p)

def sigma_of_p(p):
    p = np.clip(p, 1e-9, 1-1e-9)
    return sigma_eff * p*(1-p)

def dL_dp(p):
    p = np.clip(p, 1e-9, 1-1e-9)
    # derivative of L wrt p for affine contrast is B + noise-terms derivative
    return B + 2*(-sigma_P**2 - sigma_Q**2 + 2*rho*sigma_P*sigma_Q)

def db_dp(p):
    p = np.clip(p, 1e-9, 1-1e-9)
    # b(p) = p(1-p)L(p); derivative: (1-2p)L(p) + p(1-p)L'(p)
    return (1 - 2*p)*L_of_p(p) + p*(1-p)*dL_dp(p)

def dsigma_dp(p):
    p = np.clip(p, 1e-9, 1-1e-9)
    return sigma_eff*(1 - 2*p)

# Simulation parameters
N_paths = 1500              # number of Monte Carlo paths (keep moderate for speed)
p0 = 0.10                   # initial prevalence
rng = np.random.default_rng(12)

# Precompute Brownian increments for reproducibility
dW = rng.normal(0.0, np.sqrt(dt), size=(N_paths, n_steps))  # standard increments

# ---------------------- Simulate prevalence paths ----------------------
p_paths = np.empty((N_paths, n_steps+1), dtype=float)
p_paths[:, 0] = p0

for k in range(n_steps):
    p = p_paths[:, k]
    # Euler–Maruyama step
    drift = b_of_p(p)
    sig = sigma_of_p(p)
    p_next = p + drift*dt + sig*dW[:, k]
    # project softly back to (0,1) to ensure numerical stability
    p_next = np.clip(p_next, 1e-9, 1-1e-9)
    p_paths[:, k+1] = p_next

# ---------------------- Grid of (s, t) pairs ----------------------
# choose a coarse grid to keep computation manageable
n_s = 30
n_t = 30
s_grid_idx = np.linspace(0, n_steps-1, n_s, dtype=int)
t_grid_idx = np.linspace(1, n_steps, n_t, dtype=int)

E_absD = np.zeros((n_t, n_s))    # E[ |D_s p(t)| ]  (rows=t, cols=s)
Corr_Dp = np.full((n_t, n_s), np.nan)  # Corr(D_s p(t), p(t))

# ---------------------- Compute Malliavin derivative via tangent SDE ----------------------
# For each s in s_grid, we compute Z for all t >= s using same Brownian increments.
for j, s_idx in enumerate(s_grid_idx):
    # initial Z at s: sigma(p_s)
    p_s = p_paths[:, s_idx]
    Z = sigma_of_p(p_s).copy()  # shape (N_paths,)
    # We'll propagate forward, saving snapshots at t_grid_idx as needed
    next_t_positions = {t_idx: i for i, t_idx in enumerate(t_grid_idx) if t_idx >= s_idx}
    # We'll accumulate numbers in temp arrays per t for later stats
    # To avoid storing all Z along time, compute on the fly and record at t's of interest.
    # Create a dict: t_idx -> (list of Z, list of p(t))
    # But to keep memory small, we'll fill E_absD and Corr online by storing sums.
    # For correlation we need mean(Z), mean(p), mean(Z p), so keep three sums.
    sums = {t_idx: {'sum_absZ': 0.0, 'sumZ': 0.0, 'sumP': 0.0, 'sumZP': 0.0, 'sumZ2': 0.0, 'sumP2': 0.0, 'count': 0}
            for t_idx in t_grid_idx if t_idx >= s_idx}

    for k in range(s_idx, n_steps):
        # advance Z one step using the same Brownian increment dW[:, k]
        p_curr = p_paths[:, k]
        dZ_drift = db_dp(p_curr) * Z
        dZ_diff  = dsigma_dp(p_curr) * Z
        Z = Z + dZ_drift*dt + dZ_diff*dW[:, k]
        # If k+1 is a requested t index, record stats
        t_idx_candidate = k+1
        if t_idx_candidate in sums:
            p_t = p_paths[:, t_idx_candidate]
            S = sums[t_idx_candidate]
            # accumulate
            S['sum_absZ'] += np.sum(np.abs(Z))
            S['sumZ'] += np.sum(Z)
            S['sumP'] += np.sum(p_t)
            S['sumZP'] += np.sum(Z * p_t)
            S['sumZ2'] += np.sum(Z * Z)
            S['sumP2'] += np.sum(p_t * p_t)
            S['count'] += Z.shape[0]

    # Fill heatmaps along this column (fixed s)
    for i, t_idx in enumerate(t_grid_idx):
        if t_idx < s_idx:
            # undefined region (t < s): leave as NaN
            continue
        S = sums[t_idx]
        cnt = S['count']
        if cnt > 0:
            mean_absZ = S['sum_absZ'] / cnt
            E_absD[i, j] = mean_absZ
            # correlation
            meanZ = S['sumZ'] / cnt
            meanP = S['sumP'] / cnt
            covZP = S['sumZP']/cnt - meanZ*meanP
            varZ = S['sumZ2']/cnt - meanZ*meanZ
            varP = S['sumP2']/cnt - meanP*meanP
            if varZ > 1e-14 and varP > 1e-14:
                Corr_Dp[i, j] = covZP / np.sqrt(varZ*varP)

# Mask lower-triangle where t < s for plotting clarity
mask = np.zeros_like(E_absD, dtype=bool)
for i, t_idx in enumerate(t_grid_idx):
    for j, s_idx in enumerate(s_grid_idx):
        if t_idx < s_idx:
            mask[i, j] = True

E_absD_masked = np.ma.array(E_absD, mask=mask)
Corr_Dp_masked = np.ma.array(Corr_Dp, mask=mask)

# ---------------------- Plotting helpers ----------------------
outdir = "/content/Data/ihm_figs"
os.makedirs(outdir, exist_ok=True)

def plot_heatmap(Z, title, fname_png, fname_pdf, vmin=None, vmax=None, cmap='viridis'):
    fig, ax = plt.subplots(figsize=(7, 5))
    im = ax.imshow(Z, origin='lower', aspect='auto',
                   extent=[times[s_grid_idx[0]], times[s_grid_idx[-1]], times[t_grid_idx[0]], times[t_grid_idx[-1]]],
                   vmin=vmin, vmax=vmax, cmap=cmap)
    ax.set_xlabel(r"$s$ (conditioning time)")
    ax.set_ylabel(r"$t$ (evaluation time)")
    ax.set_title(title, fontsize=11)
    cbar = plt.colorbar(im, ax=ax)
    cbar.ax.set_ylabel("value", rotation=90, labelpad=10)
    # overlay a diagonal t=s line within the plotted extent
    s0, s1 = times[s_grid_idx[0]], times[s_grid_idx[-1]]
    t0, t1 = times[t_grid_idx[0]], times[t_grid_idx[-1]]
    line_s = np.linspace(s0, s1, 200)
    ax.plot(line_s, line_s, lw=1.0)
    fig.tight_layout()
    fig.savefig(os.path.join(outdir, fname_png), dpi=180, bbox_inches="tight")
    fig.savefig(os.path.join(outdir, fname_pdf), bbox_inches="tight")
    plt.close(fig)

# Choose color ranges for cleaner visual
plot_heatmap(E_absD_masked, r"Mean absolute Malliavin sensitivity  $\mathbb{E}[\,|D_s p(t)|\,]$",
             "fig7_heatmap_EabsD.png", "fig7_heatmap_EabsD.pdf",
             vmin=np.nanmin(E_absD_masked), vmax=np.nanpercentile(E_absD_masked, 99))

plot_heatmap(Corr_Dp_masked, r"Correlation  $\mathrm{Corr}(D_s p(t),\, p(t))$",
             "fig8_heatmap_corr.png", "fig8_heatmap_corr.pdf",
             vmin=-1.0, vmax=1.0, cmap='coolwarm')

# Return paths for download
print("Saved:")
print(os.path.join(outdir, "fig7_heatmap_EabsD.png"))
print(os.path.join(outdir, "fig7_heatmap_EabsD.pdf"))
print(os.path.join(outdir, "fig8_heatmap_corr.png"))
print(os.path.join(outdir, "fig8_heatmap_corr.pdf"))


/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:4968: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  arr.partition(


Saved:
/content/Data/ihm_figs/fig7_heatmap_EabsD.png
/content/Data/ihm_figs/fig7_heatmap_EabsD.pdf
/content/Data/ihm_figs/fig8_heatmap_corr.png
/content/Data/ihm_figs/fig8_heatmap_corr.pdf


In [ ]:
# Lighter Monte Carlo so it runs quickly in this environment.
import numpy as np
import pandas as pd
from math import sqrt
import os
# from caas_jupyter_tools import display_dataframe_to_user

T = 10.0
N = 600
M = 6000
p0 = 0.1
A = 0.05
B = 0.30
sigma_P = 0.20
sigma_Q = 0.15
rho_list = [-0.5, 0.0, 0.5, 0.9]

rng = np.random.default_rng(123)
dt = T / N
sqrt_dt = sqrt(dt)
eps = 1e-8

def simulate_one_rho(rho):
    sigma_eff = sqrt(sigma_P**2 + sigma_Q**2 - 2.0 * rho * sigma_P * sigma_Q)
    C0 = A + 2*sigma_P**2 - 2*rho*sigma_P*sigma_Q
    C1 = B - 2*sigma_P**2 - 2*sigma_Q**2 + 4*rho*sigma_P*sigma_Q

    dW = rng.normal(0.0, sqrt_dt, size=(M, N))

    p = np.full((M, N+1), p0, dtype=float)
    for n in range(N):
        pn = p[:, n]
        b = pn*(1.0 - pn)*(C0 + C1*pn)
        sig = sigma_eff * pn*(1.0 - pn)
        p[:, n+1] = pn + b*dt + sig * dW[:, n]
        # clamp
        p_next = p[:, n+1]
        p_next[p_next <= eps] = eps
        p_next[p_next >= 1.0-eps] = 1.0 - eps

    s_idx = N//2

    p_slice = p[:, s_idx:]
    pn = p_slice[:, :-1]
    dW_slice = dW[:, s_idx:]

    bprime = (1.0 - 2.0*pn)*(C0 + C1*pn) + pn*(1.0 - pn)*C1
    sigma_eff_const = sigma_eff
    sigprime = sigma_eff_const * (1.0 - 2.0*pn)

    det_int = np.sum(bprime * dt - 0.5*(sigprime**2)*dt, axis=1)
    stoch_int = np.sum(sigprime * dW_slice, axis=1)
    exponent = det_int + stoch_int

    p_s = p[:, s_idx]
    D = sigma_eff_const * p_s * (1.0 - p_s) * np.exp(exponent)

    pT = p[:, -1]

    def mc_stats(x):
        mean = x.mean()
        var = x.var(ddof=1)
        se = np.sqrt(var / M)
        ci_lo = mean - 1.96*se
        ci_hi = mean + 1.96*se
        return mean, var, se, ci_lo, ci_hi

    mean_pT, var_pT, se_mean_pT, ci_lo_pT, ci_hi_pT = mc_stats(pT)
    mean_absD, var_absD, se_mean_absD, ci_lo_absD, ci_hi_absD = mc_stats(np.abs(D))

    cov = np.cov(D, pT, ddof=1)
    corr = cov[0,1] / np.sqrt(cov[0,0]*cov[1,1])

    return {
        "rho": rho,
        "sigma_eff": sigma_eff,
        "C0": C0,
        "C1": C1,
        "E[p(T)]": mean_pT,
        "SE(E[p(T)])": se_mean_pT,
        "95% CI E[p(T)] lo": ci_lo_pT,
        "95% CI E[p(T)] hi": ci_hi_pT,
        "Var[p(T)]": var_pT,
        "E[|D_s p(T)|]": mean_absD,
        "SE(E|D|)": se_mean_absD,
        "95% CI E|D| lo": ci_lo_absD,
        "95% CI E|D| hi": ci_hi_absD,
        "Corr(D_s p(T), p(T))": corr,
        "s/T": (s_idx*dt)/T,
        "T": T,
        "N": N,
        "M": M,
        "A": A,
        "B": B,
        "sigma_P": sigma_P,
        "sigma_Q": sigma_Q
    }

rows = [simulate_one_rho(rho) for rho in rho_list]
df = pd.DataFrame(rows)

cols = [
    "rho", "sigma_eff", "C0", "C1",
    "E[p(T)]", "SE(E[p(T)])", "95% CI E[p(T)] lo", "95% CI E[p(T)] hi", "Var[p(T)]",
    "E[|D_s p(T)|]", "SE(E|D|)", "95% CI E|D| lo", "95% CI E|D| hi",
    "Corr(D_s p(T), p(T))",
    "s/T", "T", "N", "M", "A", "B", "sigma_P", "sigma_Q"
]
df = df[cols]

# Save to CSV and display
os.makedirs("/content/steeven/ihm_tables", exist_ok=True)
csv_path = "/content/steeven/ihm_tables/table_IHM_MC_summary.csv"
df.to_csv(csv_path, index=False)

# display_dataframe_to_user("IHM Monte Carlo summary (pT & Malliavin)", df)

print(f"Saved CSV to: {csv_path}")


Saved CSV to: /content/steeven/ihm_tables/table_IHM_MC_summary.csv


In [ ]:
# Minimal rerun: keep computations lighter and avoid external dependencies issues.
import os, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Ensure dirs
base_dir = Path("/content/data4")
(tbl_dir := base_dir / "ihm_tables").mkdir(parents=True, exist_ok=True)
(fig_dir := base_dir / "ihm_figs").mkdir(parents=True, exist_ok=True)

# Params
T = 60.0
N = 300
dt = T / N
tgrid = np.linspace(0.0, T, N+1)
s_index = N // 2
s_time = tgrid[s_index]

A = 0.05
B = 0.30
sigma_P = 0.20
sigma_Q = 0.15
rho_grid = [-0.5, 0.0, 0.5, 0.9]
M = 1500
p0 = 0.10

rng = np.random.default_rng(7)

def sigma_eff(sP, sQ, rho):
    return math.sqrt(sP**2 + sQ**2 - 2.0 * rho * sP * sQ)

def simulate_paths(rho):
    seff = sigma_eff(sigma_P, sigma_Q, rho)
    C0 = A + 2*sigma_P**2 - 2*rho*sigma_P*sigma_Q
    C1 = B - 2*sigma_P**2 - 2*sigma_Q**2 + 4*rho*sigma_P*sigma_Q
    N1 = -2*sigma_P**2 - 2*sigma_Q**2 + 4*rho*sigma_P*sigma_Q

    p = np.empty((M, N+1), dtype=float)
    p[:, 0] = p0
    dW = rng.normal(0.0, math.sqrt(dt), size=(M, N))
    eps = 1e-6
    for k in range(N):
        pk = p[:, k]
        drift = pk * (1.0 - pk) * (C0 + C1 * pk)
        diff = seff * pk * (1.0 - pk)
        p[:, k+1] = np.clip(pk + drift * dt + diff * dW[:, k], eps, 1.0 - eps)

    pk = p[:, s_index:-1]
    dW_seg = dW[:, s_index:]
    xi = (A + B*pk) + 2.0*(sigma_P**2*(1.0 - pk) - sigma_Q**2*pk) + 2.0*rho*sigma_P*sigma_Q*(2.0*pk - 1.0)
    dp_b = (1.0 - 2.0*pk)*xi + (pk*(1.0 - pk))*(B + N1)
    term1 = np.sum(dp_b * dt - 0.5*(seff**2)*(1.0 - 2.0*pk)**2 * dt, axis=1)
    term2 = np.sum(seff * (1.0 - 2.0*pk) * dW_seg, axis=1)
    D_signed = seff * p[:, s_index] * (1.0 - p[:, s_index]) * np.exp(term1 + term2)
    return p, D_signed, seff, C0, C1

rows = []
for rho in rho_grid:
    p_paths, D_signed, seff, C0, C1 = simulate_paths(rho)
    pT = p_paths[:, -1]
    D_abs = np.abs(D_signed)
    # Stats
    def stats(x):
        m = float(np.mean(x)); s = float(np.std(x, ddof=1)); se = s/np.sqrt(len(x))
        return m, se, float(s*s)
    mp, sep, varp = stats(pT)
    mD, seD, varD = stats(D_abs)
    corr = float(np.corrcoef(pT, D_signed)[0,1])
    rows.append({"rho": rho, "sigma_eff": seff, "C0": C0, "C1": C1,
                 "E[p(T)]": mp, "SE[p(T)]": sep, "Var[p(T)]": varp,
                 "E[|D_s p(T)|]": mD, "SE[|D_s p(T)|]": seD,
                 "Corr(D_s p(T), p(T))": corr, "M": M, "N": N, "T_days": T, "s_days": s_time})
    # Save quick figs
    idx = rng.choice(M, size=15, replace=False)
    fig, ax = plt.subplots(figsize=(6,4))
    for i in idx:
        ax.plot(tgrid, p_paths[i], linewidth=1.0, alpha=0.7)
    ax.set_xlabel("Time (days)"); ax.set_ylabel("Prevalence p(t)")
    ax.set_title(f"Prevalence paths (rho={rho:.1f}, sigma_eff={seff:.3f})")
    fig.tight_layout()
    fig.savefig(fig_dir / f"paths_rho_{rho:.1f}.png", dpi=180)
    fig.savefig(fig_dir / f"paths_rho_{rho:.1f}.pdf")
    plt.close(fig)

    # Heatmap (2D hist) of (p(T), |D_s p(T)|)
    H, xedges, yedges = np.histogram2d(pT, D_abs, bins=(40, 40))
    fig2, ax2 = plt.subplots(figsize=(6,4.5))
    im = ax2.imshow(np.log1p(H).T, origin="lower",
                    extent=[xedges[0], xedges[-1], yedges[0], yedges[-1]],
                    aspect="auto")
    ax2.set_xlabel("p(T)"); ax2.set_ylabel(r"|D_s p(T)|  (s = T/2)")
    ax2.set_title(f"Joint: p(T) vs |D_s p(T)| (rho={rho:.1f})")
    fig2.colorbar(im, ax=ax2, label="log(1 + counts)")
    fig2.tight_layout()
    fig2.savefig(fig_dir / f"heatmap_pT_Dabs_rho_{rho:.1f}.png", dpi=180)
    fig2.savefig(fig_dir / f"heatmap_pT_Dabs_rho_{rho:.1f}.pdf")
    plt.close(fig2)

summary_df = pd.DataFrame(rows)
summary_path = tbl_dir / "IHM_MC_summary_days.csv"
summary_df.to_csv(summary_path, index=False)

# # Display the table
# import caas_jupyter_tools
# caas_jupyter_tools.display_dataframe_to_user("IHM Monte Carlo summary (days scale)", summary_df)

print("TABLE:", summary_path.as_posix())
print("FIGS DIR:", fig_dir.as_posix())


TABLE: /content/data4/ihm_tables/IHM_MC_summary_days.csv
FIGS DIR: /content/data4/ihm_figs


In [ ]:
# === IHM Monte Carlo: publication-ready figures ===
import os, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# ---------- Output folders ----------
base_dir = Path("./ihm_outputs")  # change if you like
tbl_dir = base_dir / "tables"
fig_dir = base_dir / "figs"
tbl_dir.mkdir(parents=True, exist_ok=True)
fig_dir.mkdir(parents=True, exist_ok=True)

# ---------- Matplotlib style (journal-ish) ----------
plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 300,
    "font.size": 12,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "legend.fontsize": 11,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "axes.grid": True,
    "grid.alpha": 0.2,
    "grid.linestyle": "--",
    "axes.spines.top": False,
    "axes.spines.right": False
})

# ---------- Simulation params ----------
T = 60.0        # days
N = 300         # time steps (dt ~ 0.2 day)
dt = T / N
tgrid = np.linspace(0.0, T, N+1)
s_index = N // 2
s_time = tgrid[s_index]

# Affine contrast parameters (example)
A = 0.05
B = 0.30

# Noise on P and Q, and tested correlations
sigma_P = 0.20
sigma_Q = 0.15
rho_grid = [-0.5, 0.0, 0.5, 0.9]

# Monte Carlo size and number of plotted paths
M = 5000        # total simulated paths
PLOT_K = 100     # how many paths to draw on the line plots

# Initial prevalence
p0 = 0.10

# Random generator (reproducible)
rng = np.random.default_rng(7)

# ---------- Helpers ----------
def sigma_eff(sP, sQ, rho):
    return math.sqrt(sP**2 + sQ**2 - 2.0 * rho * sP * sQ)

def simulate_paths(rho, M, store_all=True):
    """Simulate prevalence paths and D_s p(T) via Euler–Maruyama under affine contrast."""
    seff = sigma_eff(sigma_P, sigma_Q, rho)
    C0 = A + 2*sigma_P**2 - 2*rho*sigma_P*sigma_Q
    C1 = B - 2*sigma_P**2 - 2*sigma_Q**2 + 4*rho*sigma_P*sigma_Q
    N1 = -2*sigma_P**2 - 2*sigma_Q**2 + 4*rho*sigma_P*sigma_Q  # used in ∂_p b

    p = np.empty((M, N+1), dtype=float)
    p[:, 0] = p0

    # Brownian increments
    dW = rng.normal(0.0, math.sqrt(dt), size=(M, N))

    # EM update with projection to (0,1)
    eps = 1e-6
    for k in range(N):
        pk = p[:, k]
        drift = pk * (1.0 - pk) * (C0 + C1 * pk)
        diff = seff * pk * (1.0 - pk)
        p[:, k+1] = np.clip(pk + drift * dt + diff * dW[:, k], eps, 1.0 - eps)

    # Malliavin D_s p(T), s = T/2
    pk = p[:, s_index:-1]
    dW_seg = dW[:, s_index:]
    xi = (A + B*pk) + 2.0*(sigma_P**2*(1.0 - pk) - sigma_Q**2*pk) + 2.0*rho*sigma_P*sigma_Q*(2.0*pk - 1.0)
    dp_b = (1.0 - 2.0*pk)*xi + (pk*(1.0 - pk))*(B + N1)

    term1 = np.sum(dp_b * dt - 0.5*(seff**2)*(1.0 - 2.0*pk)**2 * dt, axis=1)
    term2 = np.sum(seff * (1.0 - 2.0*pk) * dW_seg, axis=1)
    D_signed = seff * p[:, s_index] * (1.0 - p[:, s_index]) * np.exp(term1 + term2)

    return p, D_signed, seff, C0, C1

# ---------- Run simulations per rho ----------
rows = []
for rho in rho_grid:
    p_paths, D_signed, seff, C0, C1 = simulate_paths(rho, M)

    pT = p_paths[:, -1]
    D_abs = np.abs(D_signed)

    # Summary stats
    def stats(x):
        m = float(np.mean(x))
        s = float(np.std(x, ddof=1))
        se = s / np.sqrt(len(x))
        return m, se, float(s*s)

    mp, sep, varp = stats(pT)
    mD, seD, varD = stats(D_abs)
    corr = float(np.corrcoef(pT, D_signed)[0, 1])

    rows.append({
        "rho": rho, "sigma_eff": seff, "C0": C0, "C1": C1,
        "E[p(T)]": mp, "SE[p(T)]": sep, "Var[p(T)]": varp,
        "E[|D_s p(T)|]": mD, "SE[|D_s p(T)|]": seD,
        "Corr(D_s p(T), p(T))": corr, "M": M, "N": N,
        "T_days": T, "s_days": s_time
    })

    # ===== Figure 1: Ensemble prevalence (mean ± SD) + a few paths =====
    mean_path = p_paths.mean(axis=0)
    std_path = p_paths.std(axis=0, ddof=1)

    fig, ax = plt.subplots(figsize=(6.4, 4.4))
    # a few light sample paths
    idx = rng.choice(M, size=min(PLOT_K, M), replace=False)
    for i in idx:
        ax.plot(tgrid, p_paths[i], lw=0.8, alpha=0.35)

    # mean and ribbon
    ax.plot(tgrid, mean_path, lw=2.0, label="Mean prevalence")
    ax.fill_between(tgrid, mean_path - std_path, mean_path + std_path,
                    alpha=0.18, label="±1 SD")

    ax.set_xlabel("Time (days)")
    ax.set_ylabel("Prevalence $p(t)$")
    ax.set_title(f"Ensemble prevalence (ρ={rho:.1f}, σ_eff={seff:.3f})")
    ax.set_ylim(0, 1)
    ax.legend(frameon=False, ncol=2)
    fig.tight_layout()
    for ext in ("png", "pdf", "svg"):
        fig.savefig(fig_dir / f"ensemble_paths_rho_{rho:.1f}.{ext}")
    plt.close(fig)

    # ===== Figure 2: Joint heatmap (hexbin) of (p(T), |D_s p(T)|) =====
    fig2, ax2 = plt.subplots(figsize=(6.4, 4.6))
    hb = ax2.hexbin(pT, D_abs, gridsize=45, bins='log', mincnt=1)
    ax2.set_xlabel(r"$p(T)$")
    ax2.set_ylabel(r"$|D_s p(T)|$ (with $s=T/2$)")
    ax2.set_title(f"Joint density: $p(T)$ vs $|D_s p(T)|$ (ρ={rho:.1f})")
    cb = fig2.colorbar(hb, ax=ax2)
    cb.set_label("log(count)")
    fig2.tight_layout()
    for ext in ("png", "pdf", "svg"):
        fig2.savefig(fig_dir / f"hexbin_pT_Dabs_rho_{rho:.1f}.{ext}")
    plt.close(fig2)

# ---------- Save summary table ----------
summary_df = pd.DataFrame(rows)
summary_path = tbl_dir / "IHM_MC_summary_days.csv"
summary_df.to_csv(summary_path, index=False)

print("Wrote table:", summary_path.resolve())
print("Wrote figures to:", fig_dir.resolve())


Wrote table: /content/ihm_outputs/tables/IHM_MC_summary_days.csv
Wrote figures to: /content/ihm_outputs/figs


In [ ]:
# === IHM Monte Carlo: publication-ready figures ===
import os, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# ---------- Output folders ----------
base_dir = Path("/content/data2/ihm_outputs2")  # ensure downloadable in sandbox
tbl_dir = base_dir / "tables"
fig_dir = base_dir / "figs"
tbl_dir.mkdir(parents=True, exist_ok=True)
fig_dir.mkdir(parents=True, exist_ok=True)

# ---------- Matplotlib style (journal-ish) ----------
plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 300,
    "font.size": 12,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "legend.fontsize": 11,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "axes.grid": True,
    "grid.alpha": 0.2,
    "grid.linestyle": "--",
    "axes.spines.top": False,
    "axes.spines.right": False
})

# ---------- Simulation params ----------
T = 60.0        # days
N = 300         # time steps (dt ~ 0.2 day)
dt = T / N
tgrid = np.linspace(0.0, T, N+1)
s_index = N // 2
s_time = tgrid[s_index]

# Affine contrast parameters (example)
A = 0.05
B = 0.30

# Noise on P and Q, and tested correlations
sigma_P = 0.20
sigma_Q = 0.15
rho_grid = [-0.5, 0.0, 0.5, 0.9]

# Monte Carlo size and number of plotted paths
M = 5000        # total simulated paths
PLOT_K = 20     # how many paths to draw on the line plots

# Initial prevalence
p0 = 0.10

# Random generator (reproducible)
rng = np.random.default_rng(7)

# ---------- Helpers ----------
def sigma_eff(sP, sQ, rho):
    return math.sqrt(sP**2 + sQ**2 - 2.0 * rho * sP * sQ)

def simulate_paths(rho, M, store_all=True):
    """Simulate prevalence paths and D_s p(T) via Euler–Maruyama under affine contrast."""
    seff = sigma_eff(sigma_P, sigma_Q, rho)
    C0 = A + 2*sigma_P**2 - 2*rho*sigma_P*sigma_Q
    C1 = B - 2*sigma_P**2 - 2*sigma_Q**2 + 4*rho*sigma_P*sigma_Q
    N1 = -2*sigma_P**2 - 2*sigma_Q**2 + 4*rho*sigma_P*sigma_Q  # used in ∂_p b

    p = np.empty((M, N+1), dtype=float)
    p[:, 0] = p0

    # Brownian increments
    dW = rng.normal(0.0, math.sqrt(dt), size=(M, N))

    # EM update with projection to (0,1)
    eps = 1e-6
    for k in range(N):
        pk = p[:, k]
        drift = pk * (1.0 - pk) * (C0 + C1 * pk)
        diff = seff * pk * (1.0 - pk)
        p[:, k+1] = np.clip(pk + drift * dt + diff * dW[:, k], eps, 1.0 - eps)

    # Malliavin D_s p(T), s = T/2
    pk = p[:, s_index:-1]
    dW_seg = dW[:, s_index:]
    xi = (A + B*pk) + 2.0*(sigma_P**2*(1.0 - pk) - sigma_Q**2*pk) + 2.0*rho*sigma_P*sigma_Q*(2.0*pk - 1.0)
    dp_b = (1.0 - 2.0*pk)*xi + (pk*(1.0 - pk))*(B + N1)

    term1 = np.sum(dp_b * dt - 0.5*(seff**2)*(1.0 - 2.0*pk)**2 * dt, axis=1)
    term2 = np.sum(seff * (1.0 - 2.0*pk) * dW_seg, axis=1)
    D_signed = seff * p[:, s_index] * (1.0 - p[:, s_index]) * np.exp(term1 + term2)

    return p, D_signed, seff, C0, C1

# ---------- Run simulations per rho ----------
rows = []
for rho in rho_grid:
    p_paths, D_signed, seff, C0, C1 = simulate_paths(rho, M)

    pT = p_paths[:, -1]
    D_abs = np.abs(D_signed)

    # Summary stats
    def stats(x):
        m = float(np.mean(x))
        s = float(np.std(x, ddof=1))
        se = s / np.sqrt(len(x))
        return m, se, float(s*s)

    mp, sep, varp = stats(pT)
    mD, seD, varD = stats(D_abs)
    corr = float(np.corrcoef(pT, D_signed)[0, 1])

    rows.append({
        "rho": rho, "sigma_eff": seff, "C0": C0, "C1": C1,
        "E[p(T)]": mp, "SE[p(T)]": sep, "Var[p(T)]": varp,
        "E[|D_s p(T)|]": mD, "SE[|D_s p(T)|]": seD,
        "Corr(D_s p(T), p(T))": corr, "M": M, "N": N,
        "T_days": T, "s_days": s_time
    })

    # ===== Figure 1: Ensemble prevalence (mean ± SD) + a few paths =====
    mean_path = p_paths.mean(axis=0)
    std_path = p_paths.std(axis=0, ddof=1)

    fig, ax = plt.subplots(figsize=(6.4, 4.4))
    # a few light sample paths
    idx = rng.choice(M, size=min(PLOT_K, M), replace=False)
    for i in idx:
        ax.plot(tgrid, p_paths[i], lw=0.8, alpha=0.35)

    # mean and ribbon
    ax.plot(tgrid, mean_path, lw=2.0, label="Mean prevalence")
    ax.fill_between(tgrid, mean_path - std_path, mean_path + std_path,
                    alpha=0.18, label="±1 SD")

    ax.set_xlabel("Time (days)")
    ax.set_ylabel("Prevalence $p(t)$")
    ax.set_title(f"Ensemble prevalence (ρ={rho:.1f}, σ_eff={seff:.3f})")
    ax.set_ylim(0, 1)
    ax.legend(frameon=False, ncol=2)
    fig.tight_layout()
    for ext in ("png", "pdf", "svg"):
        fig.savefig(fig_dir / f"ensemble_paths_rho_{rho:.1f}.{ext}")
    plt.close(fig)

    # ===== Figure 2: Joint heatmap (hexbin) of (p(T), |D_s p(T)|) =====
    fig2, ax2 = plt.subplots(figsize=(6.4, 4.6))
    hb = ax2.hexbin(pT, D_abs, gridsize=45, bins='log', mincnt=1)
    ax2.set_xlabel(r"$p(T)$")
    ax2.set_ylabel(r"$|D_s p(T)|$ (with $s=T/2$)")
    ax2.set_title(f"Joint density: $p(T)$ vs $|D_s p(T)|$ (ρ={rho:.1f})")
    cb = fig2.colorbar(hb, ax=ax2)
    cb.set_label("log(count)")
    fig2.tight_layout()
    for ext in ("png", "pdf", "svg"):
        fig2.savefig(fig_dir / f"hexbin_pT_Dabs_rho_{rho:.1f}.{ext}")
    plt.close(fig2)

# ---------- Save summary table ----------
summary_df = pd.DataFrame(rows)
summary_path = tbl_dir / "IHM_MC_summary_days.csv"
summary_df.to_csv(summary_path, index=False)

summary_path.as_posix(), fig_dir.as_posix()


('/content/data2/ihm_outputs2/tables/IHM_MC_summary_days.csv',
 '/content/data2/ihm_outputs2/figs')

In [ ]:
# Panel figures for IHM Monte Carlo across rho values (ensemble + heatmap)
import os, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# ---------- Output folders ----------
base_dir = Path("/content/DATA/ihm_outputs")  # persistent in sandbox
tbl_dir = base_dir / "tables"
fig_dir = base_dir / "figs"
tbl_dir.mkdir(parents=True, exist_ok=True)
fig_dir.mkdir(parents=True, exist_ok=True)

# ---------- Matplotlib style ----------
plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 300,
    "font.size": 12,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "legend.fontsize": 10,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "grid.linestyle": "--",
    "axes.spines.top": False,
    "axes.spines.right": False
})

# ---------- Simulation params ----------
T = 60.0        # days
N = 300         # time steps (dt ~ 0.2 day)
dt = T / N
tgrid = np.linspace(0.0, T, N+1)
s_index = N // 2
s_time = tgrid[s_index]

# Affine contrast parameters (example)
A = 0.05
B = 0.30

# Noise on P and Q, and tested correlations
sigma_P = 0.20
sigma_Q = 0.15
rho_grid = [-0.5, 0.0, 0.5, 0.9]

# Monte Carlo size and number of plotted paths
M = 10000        # total simulated paths
PLOT_K = 50     # how many paths to draw on the line plots

# Initial prevalence
p0 = 0.10

# Random generator (reproducible)
rng = np.random.default_rng(77)

# ---------- Helpers ----------
def sigma_eff(sP, sQ, rho):
    return math.sqrt(sP**2 + sQ**2 - 2.0 * rho * sP * sQ)

def simulate_paths(rho, M, store_all=True):
    """Simulate prevalence paths and D_s p(T) via Euler–Maruyama under affine contrast."""
    seff = sigma_eff(sigma_P, sigma_Q, rho)
    C0 = A + 2*sigma_P**2 - 2*rho*sigma_P*sigma_Q
    C1 = B - 2*sigma_P**2 - 2*sigma_Q**2 + 4*rho*sigma_P*sigma_Q
    N1 = -2*sigma_P**2 - 2*sigma_Q**2 + 4*rho*sigma_P*sigma_Q  # used in ∂_p b

    p = np.empty((M, N+1), dtype=float)
    p[:, 0] = p0

    # Brownian increments
    dW = rng.normal(0.0, math.sqrt(dt), size=(M, N))

    # EM update with projection to (0,1)
    eps = 1e-6
    for k in range(N):
        pk = p[:, k]
        drift = pk * (1.0 - pk) * (C0 + C1 * pk)
        diff = seff * pk * (1.0 - pk)
        p[:, k+1] = np.clip(pk + drift * dt + diff * dW[:, k], eps, 1.0 - eps)

    # Malliavin D_s p(T), s = T/2
    pk = p[:, s_index:-1]
    dW_seg = dW[:, s_index:]
    xi = (A + B*pk) + 2.0*(sigma_P**2*(1.0 - pk) - sigma_Q**2*pk) + 2.0*rho*sigma_P*sigma_Q*(2.0*pk - 1.0)
    dp_b = (1.0 - 2.0*pk)*xi + (pk*(1.0 - pk))*(B + N1)

    term1 = np.sum(dp_b * dt - 0.5*(seff**2)*(1.0 - 2.0*pk)**2 * dt, axis=1)
    term2 = np.sum(seff * (1.0 - 2.0*pk) * dW_seg, axis=1)
    D_signed = seff * p[:, s_index] * (1.0 - p[:, s_index]) * np.exp(term1 + term2)

    return p, D_signed, seff, C0, C1

# ---------- Run simulations & cache for panels ----------
panel_data = {}
rows = []
for rho in rho_grid:
    p_paths, D_signed, seff, C0, C1 = simulate_paths(rho, M)
    pT = p_paths[:, -1]
    D_abs = np.abs(D_signed)

    # Summary stats
    def stats(x):
        m = float(np.mean(x))
        s = float(np.std(x, ddof=1))
        se = s / np.sqrt(len(x))
        return m, se, float(s*s)

    mp, sep, varp = stats(pT)
    mD, seD, varD = stats(D_abs)
    corr = float(np.corrcoef(pT, D_signed)[0, 1])

    rows.append({
        "rho": rho, "sigma_eff": seff, "C0": C0, "C1": C1,
        "E[p(T)]": mp, "SE[p(T)]": sep, "Var[p(T)]": varp,
        "E[|D_s p(T)|]": mD, "SE[|D_s p(T)|]": seD,
        "Corr(D_s p(T), p(T))": corr, "M": M, "N": N,
        "T_days": T, "s_days": s_time
    })
    panel_data[rho] = (p_paths, D_signed, pT, D_abs, seff)

# Save table
summary_df = pd.DataFrame(rows)
summary_path = tbl_dir / "IHM_MC_summary_days.csv"
summary_df.to_csv(summary_path, index=False)

# ---------- Panel A: 2x2 ensemble prevalence (one page) ----------
figA, axesA = plt.subplots(2, 2, figsize=(9.0, 6.6), sharex=True, sharey=True)
axesA = axesA.ravel()
for ax, rho in zip(axesA, rho_grid):
    p_paths, D_signed, pT, D_abs, seff = panel_data[rho]
    mean_path = p_paths.mean(axis=0)
    std_path = p_paths.std(axis=0, ddof=1)
    # light sample paths
    rng_local = np.random.default_rng(11)  # per-subplot reproducibility
    idx = rng_local.choice(M, size=min(PLOT_K, M), replace=False)
    for i in idx:
        ax.plot(tgrid, p_paths[i], lw=0.7, alpha=0.35)
    ax.plot(tgrid, mean_path, lw=2.0, label="Mean")
    ax.fill_between(tgrid, mean_path - std_path, mean_path + std_path, alpha=0.18, label="±1 SD")
    ax.set_title(f"ρ={rho:.1f},  σ_eff={seff:.3f}")
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.2, linestyle="--")
    if ax in (axesA[0], axesA[1]):
        ax.set_xlabel("")
    else:
        ax.set_xlabel("Time (days)")
    if ax in (axesA[1], axesA[3]):
        ax.set_ylabel("")
    else:
        ax.set_ylabel("Prevalence  $p(t)$")
# global legend
handles, labels = axesA[0].get_legend_handles_labels()
figA.legend(handles, labels, loc="lower center", ncol=4, frameon=False, bbox_to_anchor=(0.5, 0.01))
figA.tight_layout(rect=[0, 0.03, 1, 1])
panelA_png = fig_dir / "panelA_ensemble_paths_2x2.png"
panelA_pdf = fig_dir / "panelA_ensemble_paths_2x2.pdf"
panelA_svg = fig_dir / "panelA_ensemble_paths_2x2.svg"
figA.savefig(panelA_png)
figA.savefig(panelA_pdf)
figA.savefig(panelA_svg)
plt.close(figA)

# ---------- Panel B: 2x2 hexbin heatmaps (one page) ----------
figB, axesB = plt.subplots(2, 2, figsize=(9.0, 6.6), sharex=True, sharey=True)
axesB = axesB.ravel()
for ax, rho in zip(axesB, rho_grid):
    p_paths, D_signed, pT, D_abs, seff = panel_data[rho]
    hb = ax.hexbin(pT, D_abs, gridsize=45, bins='log', mincnt=1)
    ax.set_title(f"ρ={rho:.1f},  σ_eff={seff:.3f}")
    ax.grid(True, alpha=0.2, linestyle="--")
    if ax in (axesB[0], axesB[1]):
        ax.set_xlabel("")
    else:
        ax.set_xlabel(r"$p(T)$")
    if ax in (axesB[1], axesB[3]):
        ax.set_ylabel("")
    else:
        ax.set_ylabel(r"$|D_s p(T)|$  (s = T/2)")
# Shared colorbar
cax = figB.add_axes([0.92, 0.15, 0.02, 0.7])
cb = figB.colorbar(hb, cax=cax)
cb.set_label("log(count)")
figB.tight_layout(rect=[0, 0, 0.90, 1])
panelB_png = fig_dir / "panelB_hexbin_joint_2x2.png"
panelB_pdf = fig_dir / "panelB_hexbin_joint_2x2.pdf"
panelB_svg = fig_dir / "panelB_hexbin_joint_2x2.svg"
figB.savefig(panelB_png)
figB.savefig(panelB_pdf)
figB.savefig(panelB_svg)
plt.close(figB)

print("Saved table:", summary_path.as_posix())
print("Saved Panel A (ensemble) to:", panelA_png.as_posix(), panelA_pdf.as_posix(), panelA_svg.as_posix())
print("Saved Panel B (heatmaps) to:", panelB_png.as_posix(), panelB_pdf.as_posix(), panelB_svg.as_posix())


/tmp/ipython-input-3168925508.py:186: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  figB.tight_layout(rect=[0, 0, 0.90, 1])


Saved table: /content/DATA/ihm_outputs/tables/IHM_MC_summary_days.csv
Saved Panel A (ensemble) to: /content/DATA/ihm_outputs/figs/panelA_ensemble_paths_2x2.png /content/DATA/ihm_outputs/figs/panelA_ensemble_paths_2x2.pdf /content/DATA/ihm_outputs/figs/panelA_ensemble_paths_2x2.svg
Saved Panel B (heatmaps) to: /content/DATA/ihm_outputs/figs/panelB_hexbin_joint_2x2.png /content/DATA/ihm_outputs/figs/panelB_hexbin_joint_2x2.pdf /content/DATA/ihm_outputs/figs/panelB_hexbin_joint_2x2.svg


In [ ]:
# === IHM Monte Carlo: figures + CIs + convergence + dt-check + sigma_eff estimation ===
import os, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# ---------- Output folders ----------
base_dir = Path("/content/new/ihm_outputs")  # change if you like
tbl_dir = base_dir / "tables"
fig_dir = base_dir / "figs"
tbl_dir.mkdir(parents=True, exist_ok=True)
fig_dir.mkdir(parents=True, exist_ok=True)

# ---------- Matplotlib style (journal-ish) ----------
plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 300,
    "font.size": 12,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "legend.fontsize": 11,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "axes.grid": True,
    "grid.alpha": 0.2,
    "grid.linestyle": "--",
    "axes.spines.top": False,
    "axes.spines.right": False
})

# ---------- Core simulation params (days) ----------
T = 60.0        # horizon in days
N = 300         # time steps (dt ~ 0.2 day)
dt = T / N
tgrid = np.linspace(0.0, T, N+1)
s_index = N // 2                # s = T/2 for Malliavin sensitivity
s_time = tgrid[s_index]

# Affine contrast (A + B p) for examples
A = 0.05
B = 0.30

# Noise on P and Q, and tested correlations
sigma_P = 0.20
sigma_Q = 0.15
rho_grid = [-0.5, 0.0, 0.5, 0.9]

# Monte Carlo size and number of visible paths in panels
M = 10000       # <-- set large for final results (e.g., 10_000)
PLOT_K = 50     # <-- how many paths to draw on path panels
p0 = 0.10       # initial prevalence

# Random generator (reproducible)
rng = np.random.default_rng(7)

# ---------- Helpers ----------
def sigma_eff(sP, sQ, rho):
    return math.sqrt(sP**2 + sQ**2 - 2.0 * rho * sP * sQ)

def coeffs_from_rho(rho):
    """Return (seff, C0, C1, N1) for given rho under affine contrast."""
    seff = sigma_eff(sigma_P, sigma_Q, rho)
    C0 = A + 2*sigma_P**2 - 2*rho*sigma_P*sigma_Q
    C1 = B - 2*sigma_P**2 - 2*sigma_Q**2 + 4*rho*sigma_P*sigma_Q
    N1 = -2*sigma_P**2 - 2*sigma_Q**2 + 4*rho*sigma_P*sigma_Q  # appears in ∂_p b
    return seff, C0, C1, N1

def simulate_paths(rho, M, N, T, rng):
    """Simulate p-paths and D_s p(T) via Euler–Maruyama under affine contrast."""
    dt = T / N
    tgrid = np.linspace(0.0, T, N+1)
    s_index = N // 2

    seff, C0, C1, N1 = coeffs_from_rho(rho)
    p = np.empty((M, N+1), dtype=float)
    p[:, 0] = p0

    # Brownian increments
    dW = rng.normal(0.0, math.sqrt(dt), size=(M, N))

    # EM update with projection to (0,1)
    eps = 1e-6
    for k in range(N):
        pk = p[:, k]
        drift = pk * (1.0 - pk) * (C0 + C1 * pk)
        diff = seff * pk * (1.0 - pk)
        p[:, k+1] = np.clip(pk + drift * dt + diff * dW[:, k], eps, 1.0 - eps)

    # Malliavin D_s p(T), s = T/2
    pk = p[:, s_index:-1]
    dW_seg = dW[:, s_index:]
    xi = (A + B*pk) + 2.0*(sigma_P**2*(1.0 - pk) - sigma_Q**2*pk) + 2.0*rho*sigma_P*sigma_Q*(2.0*pk - 1.0)
    dp_b = (1.0 - 2.0*pk)*xi + (pk*(1.0 - pk))*(B + N1)

    term1 = np.sum(dp_b * dt - 0.5*(seff**2)*(1.0 - 2.0*pk)**2 * dt, axis=1)
    term2 = np.sum(seff * (1.0 - 2.0*pk) * dW_seg, axis=1)
    D_signed = seff * p[:, s_index] * (1.0 - p[:, s_index]) * np.exp(term1 + term2)

    return p, D_signed, seff, C0, C1, tgrid, s_index

def mean_se_var(x):
    m = float(np.mean(x))
    s = float(np.std(x, ddof=1))
    se = s / np.sqrt(len(x))
    return m, se, float(s*s), 1.96*se

# ---------- MAIN: per-rho simulations, figures, table with 95% CIs ----------
rows = []
for rho in rho_grid:
    p_paths, D_signed, seff, C0, C1, tgrid, s_index = simulate_paths(rho, M, N, T, rng)
    pT = p_paths[:, -1]
    D_abs = np.abs(D_signed)

    # Summaries (+ 95% CIs)
    mp, sep, varp, ci_p = mean_se_var(pT)
    mD, seD, varD, ci_D = mean_se_var(D_abs)
    corr = float(np.corrcoef(pT, D_signed)[0, 1])

    rows.append({
        "rho": rho, "sigma_eff": seff, "C0": C0, "C1": C1,
        "E[p(T)]": mp, "SE[p(T)]": sep, "CI95[p(T)]": ci_p,
        "Var[p(T)]": varp,
        "E[|D_s p(T)|]": mD, "SE[|D_s p(T)|]": seD, "CI95[|D_s p(T)|]": ci_D,
        "Corr(D_s p(T), p(T))": corr, "M": M, "N": N,
        "T_days": T, "s_days": tgrid[s_index]
    })

    # ===== Figure 1: Ensemble mean ± SD + sample paths + optional 95% CI for mean =====
    mean_path = p_paths.mean(axis=0)
    std_path  = p_paths.std(axis=0, ddof=1)
    se_path   = std_path / np.sqrt(M)

    fig, ax = plt.subplots(figsize=(6.6, 4.6))
    idx_plot = rng.choice(M, size=min(PLOT_K, M), replace=False)
    for i in idx_plot:
        ax.plot(tgrid, p_paths[i], lw=0.7, alpha=0.30)

    ax.plot(tgrid, mean_path, lw=2.0, label="Mean prevalence")
    ax.fill_between(tgrid, mean_path - std_path, mean_path + std_path,
                    alpha=0.18, label="±1 SD")
    ax.fill_between(tgrid, mean_path - 1.96*se_path, mean_path + 1.96*se_path,
                    alpha=0.20, color="k", ec="none", label="95% CI (mean)")

    ax.set_xlabel("Time (days)")
    ax.set_ylabel("Prevalence $p(t)$")
    ax.set_title(f"Ensemble prevalence (ρ={rho:.1f}, σ_eff={seff:.3f})")
    ax.set_ylim(0, 1)
    ax.legend(frameon=False, ncol=3)
    fig.tight_layout()
    for ext in ("png", "pdf", "svg"):
        fig.savefig(fig_dir / f"ensemble_paths_rho_{rho:.1f}.{ext}")
    plt.close(fig)

    # ===== Figure 2: Joint hexbin of (p(T), |D_s p(T)|) =====
    fig2, ax2 = plt.subplots(figsize=(6.4, 4.6))
    hb = ax2.hexbin(pT, D_abs, gridsize=50, bins='log', mincnt=1)
    ax2.set_xlabel(r"$p(T)$")
    ax2.set_ylabel(r"$|D_s p(T)|$ (with $s=T/2$)")
    ax2.set_title(f"Joint density: $p(T)$ vs $|D_s p(T)|$ (ρ={rho:.1f})")
    cb = fig2.colorbar(hb, ax=ax2)
    cb.set_label("log(count)")
    fig2.tight_layout()
    for ext in ("png", "pdf", "svg"):
        fig2.savefig(fig_dir / f"hexbin_pT_Dabs_rho_{rho:.1f}.{ext}")
    plt.close(fig2)

# ---------- Save main summary table (with CIs) ----------
summary_df = pd.DataFrame(rows)
summary_path = tbl_dir / "IHM_MC_summary_days.csv"
summary_df.to_csv(summary_path, index=False)

# ---------- Convergence mini-study (vary M) ----------
conv_rows = []
M_grid = [2000, 5000, 10000]
rho_conv = 0.5  # representative rho
for M0 in M_grid:
    p_paths, D_signed, seff, C0, C1, tgrid, s_index = simulate_paths(rho_conv, M0, N, T, rng)
    pT = p_paths[:, -1]
    D_abs = np.abs(D_signed)
    mp, sep, varp, ci_p = mean_se_var(pT)
    mD, seD, varD, ci_D = mean_se_var(D_abs)
    conv_rows.append({"M": M0, "E[p(T)]": mp, "SE[p(T)]": sep, "CI95[p(T)]": ci_p,
                      "E[|D_s p(T)|]": mD, "SE[|D_s p(T)|]": seD, "CI95[|D_s p(T)|]": ci_D})

conv_df = pd.DataFrame(conv_rows)
conv_path = tbl_dir / "MC_convergence_rho_{:.1f}.csv".format(rho_conv)
conv_df.to_csv(conv_path, index=False)

# ---------- Time-step (dt) sensitivity check ----------
rho_dt = 0.5
M_dt = 5000
# baseline N, and refined 2N
p_base, D_base, seff_b, *_ = simulate_paths(rho_dt, M_dt, N,   T, rng)
p_refi, D_refi, seff_r, *_ = simulate_paths(rho_dt, M_dt, 2*N, T, rng)
# compare endpoint summaries
def summarize(arr):
    m = float(np.mean(arr)); s = float(np.std(arr, ddof=1)); se = s/np.sqrt(len(arr)); return m, se
pT_b, pT_r = p_base[:, -1], p_refi[:, -1]
m_b, se_b  = summarize(pT_b)
m_r, se_r  = summarize(pT_r)
dt_rows = [{
    "rho": rho_dt, "M": M_dt, "N_base": N, "N_refined": 2*N,
    "E[p(T)]_base": m_b, "SE_base": se_b,
    "E[p(T)]_refined": m_r, "SE_refined": se_r,
    "abs_diff_in_SEs": abs(m_b - m_r) / max(se_b, se_r)
}]
dt_df = pd.DataFrame(dt_rows)
dt_path = tbl_dir / "dt_sensitivity_rho_{:.1f}.csv".format(rho_dt)
dt_df.to_csv(dt_path, index=False)

# ---------- Synthetic estimation of sigma_eff via Euler pseudo-likelihood ----------
# Use a single long path to mimic data; treat sigma_eff unknown; maximize w.r.t. sigma_eff^2
rho_est = 0.0
M_est  = 1      # one path (long) to show identifiability
N_est  = 1200   # finer grid in time
T_est  = 60.0
rng_est = np.random.default_rng(123)

p_est, _, seff_true, C0_e, C1_e, tgrid_e, _ = simulate_paths(rho_est, M_est, N_est, T_est, rng_est)
p_est = p_est[0]  # 1D
dt_e = T_est / N_est

# Build pseudo-likelihood for sigma_eff^2
# Δp_k ~ N(b Δt, sigma_eff^2 p_k^2(1-p_k)^2 Δt)
pk = p_est[:-1]
dp = np.diff(p_est)
mu = pk * (1.0 - pk) * (C0_e + C1_e * pk) * dt_e
vk = (pk**2) * ((1.0 - pk)**2) * dt_e   # variance factor times sigma_eff^2

# Closed-form MLE for sigma_eff^2 in heteroskedastic normal model:
sigma2_hat = float(np.mean(((dp - mu)**2) / vk))
sigma_hat = math.sqrt(max(sigma2_hat, 1e-12))

# Profile log-likelihood as a function of sigma_eff
def loglik_sigma(s):
    s2 = s*s
    return -0.5*np.sum(np.log(2.0*np.pi*s2*vk) + ((dp - mu)**2)/(s2*vk))

sig_grid = np.linspace(0.05, 0.60, 140)
ll_grid  = np.array([loglik_sigma(s) for s in sig_grid])

# Plot profile log-likelihood
fig3, ax3 = plt.subplots(figsize=(6.4, 4.6))
ax3.plot(sig_grid, ll_grid, lw=2)
ax3.axvline(seff_true, color="tab:green", ls="--", lw=1.6, label=fr"true $\sigma_{{\rm eff}}={seff_true:.3f}$")
ax3.axvline(sigma_hat, color="tab:red", ls="--", lw=1.6, label=fr"MLE $\hat\sigma_{{\rm eff}}={sigma_hat:.3f}$")
ax3.set_xlabel(r"$\sigma_{\rm eff}$")
ax3.set_ylabel("Profile log-likelihood")
ax3.set_title(r"Profile log-likelihood for $\sigma_{\rm eff}$ (synthetic)")
ax3.legend(frameon=False)
fig3.tight_layout()
for ext in ("png", "pdf", "svg"):
    fig3.savefig(fig_dir / f"profile_loglik_sigma_eff_rho_{rho_est:.1f}.{ext}")
plt.close(fig3)

# ---------- Save tables ----------
summary_df.to_csv(summary_path, index=False)
conv_df.to_csv(conv_path, index=False)
dt_df.to_csv(dt_path, index=False)

print("WROTE:")
print("  main summary:", summary_path.resolve())
print("  MC convergence:", conv_path.resolve())
print("  dt sensitivity:", dt_path.resolve())
print("  figures in:", fig_dir.resolve())


WROTE:
  main summary: /content/new/ihm_outputs/tables/IHM_MC_summary_days.csv
  MC convergence: /content/new/ihm_outputs/tables/MC_convergence_rho_0.5.csv
  dt sensitivity: /content/new/ihm_outputs/tables/dt_sensitivity_rho_0.5.csv
  figures in: /content/new/ihm_outputs/figs


In [ ]:
# === IHM Monte Carlo + Bootstrap: publication-ready panels (retry) ===
import os, math, io, textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# ---------- Output folders ----------
base_dir = Path("/content/test_data/ihm_outputs")  # change if you like
tbl_dir = base_dir / "tables"
fig_dir = base_dir / "figs"
tbl_dir.mkdir(parents=True, exist_ok=True)
fig_dir.mkdir(parents=True, exist_ok=True)

# ---------- Matplotlib style (journal-ish) ----------
plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 300,
    "font.size": 12,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "legend.fontsize": 11,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "axes.grid": True,
    "grid.alpha": 0.2,
    "grid.linestyle": "--",
    "axes.spines.top": False,
    "axes.spines.right": False
})

# ---------- Simulation params (easy to tweak) ----------
T = 60.0        # days
N = 300         # time steps (dt ~ 0.2 day)
dt = T / N
tgrid = np.linspace(0.0, T, N+1)
s_index = N // 2
s_time = tgrid[s_index]

# Affine contrast parameters (example)
A = 0.05
B = 0.30

# Noise on P and Q, and tested correlations
sigma_P = 0.20
sigma_Q = 0.15
rho_grid = [-0.5, 0.0, 0.5, 0.9]

# Monte Carlo size and number of plotted paths
M = 1500        # total simulated paths (set to 10000 if you want; might be heavier)
PLOT_K = 50     # how many paths to draw on the line plots

# Initial prevalence
p0 = 0.10

# Random generator (reproducible)
rng = np.random.default_rng(7)

# ---------- Helpers ----------
def sigma_eff(sP, sQ, rho):
    return math.sqrt(sP**2 + sQ**2 - 2.0 * rho * sP * sQ)

def simulate_paths(rho, M):
    """Simulate prevalence paths and D_s p(T) via Euler–Maruyama under affine contrast."""
    seff = sigma_eff(sigma_P, sigma_Q, rho)
    C0 = A + 2*sigma_P**2 - 2*rho*sigma_P*sigma_Q
    C1 = B - 2*sigma_P**2 - 2*sigma_Q**2 + 4*rho*sigma_P*sigma_Q
    N1 = -2*sigma_P**2 - 2*sigma_Q**2 + 4*rho*sigma_P*sigma_Q  # used in ∂_p b

    p = np.empty((M, N+1), dtype=float)
    p[:, 0] = p0

    # Brownian increments
    dW = rng.normal(0.0, math.sqrt(dt), size=(M, N))

    # EM update with projection to (0,1)
    eps = 1e-6
    for k in range(N):
        pk = p[:, k]
        drift = pk * (1.0 - pk) * (C0 + C1 * pk)
        diff = seff * pk * (1.0 - pk)
        p[:, k+1] = np.clip(pk + drift * dt + diff * dW[:, k], eps, 1.0 - eps)

    # Malliavin D_s p(T), s = T/2
    pk = p[:, s_index:-1]
    dW_seg = dW[:, s_index:]
    xi = (A + B*pk) + 2.0*(sigma_P**2*(1.0 - pk) - sigma_Q**2*pk) + 2.0*rho*sigma_P*sigma_Q*(2.0*pk - 1.0)
    dp_b = (1.0 - 2.0*pk)*xi + (pk*(1.0 - pk))*(B + N1)

    term1 = np.sum(dp_b * dt - 0.5*(seff**2)*(1.0 - 2.0*pk)**2 * dt, axis=1)
    term2 = np.sum(seff * (1.0 - 2.0*pk) * dW_seg, axis=1)
    D_signed = seff * p[:, s_index] * (1.0 - p[:, s_index]) * np.exp(term1 + term2)

    return p, D_signed, seff, C0, C1

# ---------- Run simulations per rho and save per-rho figs ----------
rows = []
per_rho_store = {}  # store for panels
for rho in rho_grid:
    p_paths, D_signed, seff, C0, C1 = simulate_paths(rho, M)

    pT = p_paths[:, -1]
    D_abs = np.abs(D_signed)

    # Summary stats
    def stats(x):
        m = float(np.mean(x))
        s = float(np.std(x, ddof=1))
        se = s / np.sqrt(len(x))
        return m, se, float(s*s)

    mp, sep, varp = stats(pT)
    mD, seD, varD = stats(D_abs)
    corr = float(np.corrcoef(pT, D_signed)[0, 1])

    rows.append({
        "rho": rho, "sigma_eff": seff, "C0": C0, "C1": C1,
        "E[p(T)]": mp, "SE[p(T)]": sep, "Var[p(T)]": varp,
        "E[|D_s p(T)|]": mD, "SE[|D_s p(T)|]": seD,
        "Corr(D_s p(T), p(T))": corr, "M": M, "N": N,
        "T_days": T, "s_days": s_time
    })

    # Store for panels
    per_rho_store[rho] = (p_paths, pT, D_abs, seff)

    # ===== Single-rho Figure 1: Ensemble prevalence (mean ± SD) + a few paths =====
    mean_path = p_paths.mean(axis=0)
    std_path = p_paths.std(axis=0, ddof=1)

    fig, ax = plt.subplots(figsize=(6.4, 4.4))
    # a few light sample paths
    idx = rng.choice(M, size=min(PLOT_K, M), replace=False)
    for i in idx:
        ax.plot(tgrid, p_paths[i], lw=0.8, alpha=0.35)

    # mean and ribbon
    ax.plot(tgrid, mean_path, lw=2.0, label="Mean prevalence")
    ax.fill_between(tgrid, mean_path - std_path, mean_path + std_path,
                    alpha=0.18, label="±1 SD")

    ax.set_xlabel("Time (days)")
    ax.set_ylabel("Prevalence $p(t)$")
    ax.set_title(f"Ensemble prevalence (ρ={rho:.1f}, σ_eff={seff:.3f})")
    ax.set_ylim(0, 1)
    ax.legend(frameon=False, ncol=2)
    fig.tight_layout()
    for ext in ("png", "pdf", "svg"):
        fig.savefig(fig_dir / f"ensemble_paths_rho_{rho:.1f}.{ext}")
    plt.close(fig)

    # ===== Single-rho Figure 2: Joint heatmap (hexbin) of (p(T), |D_s p(T)|) =====
    fig2, ax2 = plt.subplots(figsize=(6.4, 4.6))
    hb = ax2.hexbin(pT, D_abs, gridsize=45, bins='log', mincnt=1)
    ax2.set_xlabel(r"$p(T)$")
    ax2.set_ylabel(r"$|D_s p(T)|$ (with $s=T/2$)")
    ax2.set_title(f"Joint density: $p(T)$ vs $|D_s p(T)|$ (ρ={rho:.1f})")
    cb = fig2.colorbar(hb, ax=ax2)
    cb.set_label("log(count)")
    fig2.tight_layout()
    for ext in ("png", "pdf", "svg"):
        fig2.savefig(fig_dir / f"hexbin_pT_Dabs_rho_{rho:.1f}.{ext}")
    plt.close(fig2)

# ---------- Save summary table ----------
summary_df = pd.DataFrame(rows)
summary_path = tbl_dir / "IHM_MC_summary_days.csv"
summary_df.to_csv(summary_path, index=False)

# ---------- Panels: 2x2 for ensemble paths; 2x2 for hexbin ----------
# Ensure exactly 4 rhos for 2x2 layout; if more, we would generalize.
assert len(rho_grid) == 4, "Panels coded for 4 rho values (2x2)."

# Panel 1: ensemble paths
figP, axsP = plt.subplots(2, 2, figsize=(10, 7.2), sharex=True, sharey=True)
axsP = axsP.ravel()
for ax, rho in zip(axsP, rho_grid):
    p_paths, pT, D_abs, seff = per_rho_store[rho]
    mean_path = p_paths.mean(axis=0)
    std_path = p_paths.std(axis=0, ddof=1)

    idx = rng.choice(M, size=min(PLOT_K, M), replace=False)
    for i in idx:
        ax.plot(tgrid, p_paths[i], lw=0.7, alpha=0.28)

    ax.plot(tgrid, mean_path, lw=2.0, label="Mean")
    ax.fill_between(tgrid, mean_path - std_path, mean_path + std_path,
                    alpha=0.18, label="±1 SD")
    ax.set_title(fr"$\rho={rho:.1f}$, $\sigma_{{\rm eff}}={seff:.3f}$")
    ax.set_ylim(0, 1)
for ax in axsP[2:]:
    ax.set_xlabel("Time (days)")
axsP[0].set_ylabel("Prevalence $p(t)$")
axsP[2].set_ylabel("Prevalence $p(t)$")
handles, labels = axsP[0].get_legend_handles_labels()
figP.legend(handles, labels, loc="upper center", ncol=2, frameon=False)
figP.tight_layout(rect=[0,0,1,0.95])
for ext in ("png", "pdf", "svg"):
    figP.savefig(fig_dir / f"panel_ensemble_paths.{ext}")
plt.close(figP)

# Panel 2: hexbin (pT vs |Ds p(T)|)
figH, axsH = plt.subplots(2, 2, figsize=(10, 7.2), sharex=True, sharey=True)
axsH = axsH.ravel()
for ax, rho in zip(axsH, rho_grid):
    p_paths, pT, D_abs, seff = per_rho_store[rho]
    hb = ax.hexbin(pT, D_abs, gridsize=40, bins='log', mincnt=1)
    ax.set_title(fr"$\rho={rho:.1f}$, $\sigma_{{\rm eff}}={seff:.3f}$")
for ax in axsH[2:]:
    ax.set_xlabel(r"$p(T)$")
axsH[0].set_ylabel(r"$|D_s p(T)|$")
axsH[2].set_ylabel(r"$|D_s p(T)|$")
cbar = figH.colorbar(hb, ax=axsH, shrink=0.88)
cbar.set_label("log(count)")
figH.tight_layout()
for ext in ("png", "pdf", "svg"):
    figH.savefig(fig_dir / f"panel_hexbin_pT_Dabs.{ext}")
plt.close(figH)

# ---------- Synthetic estimation of sigma_eff from a single long path ----------
# Choose one rho for the estimation demo
rho_est = 0.5
seff_true = sigma_eff(sigma_P, sigma_Q, rho_est)
C0_est = A + 2*sigma_P**2 - 2*rho_est*sigma_P*sigma_Q
C1_est = B - 2*sigma_P**2 - 2*sigma_Q**2 + 4*rho_est*sigma_P*sigma_Q

# Simulate one long path for estimation (can increase N_long for smoother results)
T_long = 60.0
N_long = 1200
dt_long = T_long / N_long
t_long = np.linspace(0.0, T_long, N_long+1)
p = np.empty(N_long+1, dtype=float); p[0] = p0
eps = 1e-6
dW = rng.normal(0.0, math.sqrt(dt_long), size=N_long)
for k in range(N_long):
    pk = p[k]
    drift = pk*(1.0-pk)*(C0_est + C1_est*pk)
    diff = seff_true*pk*(1.0-pk)
    p[k+1] = np.clip(pk + drift*dt_long + diff*dW[k], eps, 1.0-eps)

dp = np.diff(p)
pk = p[:-1]
mu = pk*(1.0-pk)*(C0_est + C1_est*pk)*dt_long
vk = (pk**2 * (1.0-pk)**2) * dt_long

# Closed-form MLE for sigma_eff^2 under Euler pseudo-likelihood
sigma2_hat = float(np.mean(((dp - mu)**2) / vk))
sigma_hat = math.sqrt(max(sigma2_hat, 1e-12))

# Plot the single path + residual check
figE, axE = plt.subplots(1, 2, figsize=(11, 4.2))
axE[0].plot(t_long, p, lw=1.6)
axE[0].set_xlabel("Time (days)"); axE[0].set_ylabel("Prevalence $p(t)$")
axE[0].set_title(fr"Single path for estimation ($\rho={rho_est}$)")

resid = (dp - mu) / (np.sqrt(vk) * max(sigma_hat,1e-12))
axE[1].hist(resid, bins=40, density=True, alpha=0.7)
axE[1].set_title(r"Standardized Euler residuals (histogram)")
axE[1].set_xlabel("Residual"); axE[1].set_ylabel("Density")
figE.tight_layout()
for ext in ("png", "pdf", "svg"):
    figE.savefig(fig_dir / f"sigma_eff_estimation_path_resid_rho_{rho_est:.1f}.{ext}")
plt.close(figE)

# ---------- Bootstrap CI for sigma_eff (percentile CI) ----------
B = 500  # use 1000-2000 for final if runtime allows
rng_boot = np.random.default_rng(2025)
n = len(dp)
idx_all = np.arange(n)
boot_sighat = np.empty(B, dtype=float)

for b in range(B):
    idx = rng_boot.choice(idx_all, size=n, replace=True)
    dp_b = dp[idx]
    mu_b = mu[idx]
    vk_b = vk[idx]
    sig2_b = float(np.mean(((dp_b - mu_b)**2) / vk_b))
    boot_sighat[b] = math.sqrt(max(sig2_b, 1e-12))

ci_lo, ci_hi = np.percentile(boot_sighat, [2.5, 97.5])
se_boot = float(np.std(boot_sighat, ddof=1))

boot_df = pd.DataFrame({
    "B": [B],
    "rho": [rho_est],
    "sigma_eff_true": [seff_true],
    "sigma_eff_MLE": [sigma_hat],
    "SE_boot": [se_boot],
    "CI95_lo": [ci_lo],
    "CI95_hi": [ci_hi],
    "T_days": [T_long],
    "N_steps": [N_long],
    "dt_days": [dt_long]
})
boot_path = tbl_dir / f"sigma_eff_bootstrap_rho_{rho_est:.1f}.csv"
boot_df.to_csv(boot_path, index=False)

figb, axb = plt.subplots(figsize=(6.2, 4.4))
axb.hist(boot_sighat, bins=40, density=True, alpha=0.6)
axb.axvline(seff_true, ls="--", lw=1.6, label=fr"true $\sigma_{{\rm eff}}={seff_true:.3f}$")
axb.axvline(sigma_hat, ls="--", lw=1.6, label=fr"MLE $\hat\sigma_{{\rm eff}}={sigma_hat:.3f}$")
axb.axvline(ci_lo, color="k", ls=":", lw=1.4, label="95% bootstrap CI")
axb.axvline(ci_hi, color="k", ls=":", lw=1.4)
axb.set_xlabel(r"$\widehat{\sigma}_{\rm eff}$ (bootstrap replicates)")
axb.set_ylabel("Density")
axb.set_title(r"Bootstrap distribution of $\widehat{\sigma}_{\rm eff}$")
axb.legend(frameon=False)
figb.tight_layout()
for ext in ("png", "pdf", "svg"):
    figb.savefig(fig_dir / f"bootstrap_sigma_eff_rho_{rho_est:.1f}.{ext}")
plt.close(figb)

# ---------- Write a small LaTeX snippet to include the summary table ----------
latex_snippet = rf"""
\begin{{table}}[t]
\centering
\caption{{Monte Carlo summary across correlation levels $\rho$ (time scale in days). $M={M}$ paths, $T={T}$, $\Delta t={dt:.3f}$.}}
\label{{tab:ihm-mc-summary}}
\begin{{tabular}}{{lrrrrrrrr}}
\toprule
$\rho$ & $\sigma_\mathrm{{eff}}$ & $E[p(T)]$ & SE & Var & $E[|D_s p(T)|]$ & SE & $\mathrm{{Corr}}(D_s p(T),p(T))$ \\
\midrule
"""
for r in rows:
    latex_snippet += f"{r['rho']:>4.1f} & {r['sigma_eff']:.3f} & {r['E[p(T)]']:.3f} & {r['SE[p(T)]']:.3f} & {r['Var[p(T)]']:.3f} & {r['E[|D_s p(T)|]']:.3f} & {r['SE[|D_s p(T)|]']:.3f} & {r['Corr(D_s p(T), p(T))']:.3f} \\\\\n"
latex_snippet += r"""\bottomrule
\end{tabular}
\end{table}
"""
with open(tbl_dir / "ihm_mc_summary_table.tex", "w") as f:
    f.write(latex_snippet)

print("WROTE TABLE:", summary_path.as_posix())
print("WROTE BOOTSTRAP TABLE:", boot_path.as_posix())
print("FIGS DIR:", fig_dir.as_posix())


/tmp/ipython-input-3124484360.py:215: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  figH.tight_layout()


WROTE TABLE: /content/test_data/ihm_outputs/tables/IHM_MC_summary_days.csv
WROTE BOOTSTRAP TABLE: /content/test_data/ihm_outputs/tables/sigma_eff_bootstrap_rho_0.5.csv
FIGS DIR: /content/test_data/ihm_outputs/figs


In [ ]:
# === IHM Monte Carlo: summary + panels + adjustable colorbar placement ===
import os, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.gridspec import GridSpec
from mpl_toolkits.axes_grid1 import make_axes_locatable

# ---------- Output folders ----------
base_dir = Path("/content/final/ihm_outputs")  # change if you like
tbl_dir = base_dir / "tables"
fig_dir = base_dir / "figs"
(tbl_dir).mkdir(parents=True, exist_ok=True)
(fig_dir).mkdir(parents=True, exist_ok=True)

# ---------- Matplotlib style (journal-ish) ----------
plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 300,
    "font.size": 12,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "legend.fontsize": 11,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "axes.grid": True,
    "grid.alpha": 0.2,
    "grid.linestyle": "--",
    "axes.spines.top": False,
    "axes.spines.right": False
})

# ---------- Simulation params (feel free to edit) ----------
T = 60.0        # days
N = 300         # time steps (dt ~ 0.2 day). For step-size test we will run 2N.
dt = T / N
tgrid = np.linspace(0.0, T, N+1)
s_index = N // 2
s_time = tgrid[s_index]

# Affine contrast parameters (example)
A = 0.05
B = 0.30

# Noise on P and Q, and tested correlations
sigma_P = 0.20
sigma_Q = 0.15
rho_grid = [-0.5, 0.0, 0.5, 0.9]

# Monte Carlo size and number of plotted paths
M = 10000       # total simulated paths (increase if you like; affects runtime)
PLOT_K = 50     # how many paths to draw on the line plots

# Initial prevalence
p0 = 0.10

# Random generator (reproducible)
rng = np.random.default_rng(7)

# ---------- Helpers ----------
def sigma_eff(sP, sQ, rho):
    return math.sqrt(sP**2 + sQ**2 - 2.0 * rho * sP * sQ)

def simulate_paths(rho, M, N, A, B, sigma_P, sigma_Q, p0, rng):
    """Simulate prevalence paths and D_s p(T) via Euler–Maruyama under affine contrast.
       Returns (paths, D_signed, seff, C0, C1, grid_time, s_index)."""
    T = 60.0
    dt = T / N
    tgrid = np.linspace(0.0, T, N+1)
    s_index = N // 2

    seff = sigma_eff(sigma_P, sigma_Q, rho)
    C0 = A + 2*sigma_P**2 - 2*rho*sigma_P*sigma_Q
    C1 = B - 2*sigma_P**2 - 2*sigma_Q**2 + 4*rho*sigma_P*sigma_Q
    N1 = -2*sigma_P**2 - 2*sigma_Q**2 + 4*rho*sigma_P*sigma_Q  # used in ∂_p b

    p = np.empty((M, N+1), dtype=float)
    p[:, 0] = p0

    # Brownian increments
    dW = rng.normal(0.0, math.sqrt(dt), size=(M, N))

    # EM update with projection to (0,1)
    eps = 1e-6
    for k in range(N):
        pk = p[:, k]
        drift = pk * (1.0 - pk) * (C0 + C1 * pk)
        diff = seff * pk * (1.0 - pk)
        p[:, k+1] = np.clip(pk + drift * dt + diff * dW[:, k], eps, 1.0 - eps)

    # Malliavin D_s p(T), s = T/2
    pk = p[:, s_index:-1]
    dW_seg = dW[:, s_index:]
    xi = (A + B*pk) + 2.0*(sigma_P**2*(1.0 - pk) - sigma_Q**2*pk) + 2.0*rho*sigma_P*sigma_Q*(2.0*pk - 1.0)
    dp_b = (1.0 - 2.0*pk)*xi + (pk*(1.0 - pk))*(B + N1)

    term1 = np.sum(dp_b * dt - 0.5*(seff**2)*(1.0 - 2.0*pk)**2 * dt, axis=1)
    term2 = np.sum(seff * (1.0 - 2.0*pk) * dW_seg, axis=1)
    D_signed = seff * p[:, s_index] * (1.0 - p[:, s_index]) * np.exp(term1 + term2)

    return p, D_signed, seff, C0, C1, tgrid, s_index

def ci95(mean, sd, n):
    se = sd / np.sqrt(n)
    lo = mean - 1.96 * se
    hi = mean + 1.96 * se
    return se, lo, hi

# ---------- MAIN: per-rho summary + figures (ensemble + hexbin with colorbar) ----------
rows = []
for rho in rho_grid:
    p_paths, D_signed, seff, C0, C1, tgrid, s_index = simulate_paths(
        rho, M, N, A, B, sigma_P, sigma_Q, p0, rng
    )
    pT = p_paths[:, -1]
    D_abs = np.abs(D_signed)

    # Summary stats + CIs
    mp = float(np.mean(pT)); sp = float(np.std(pT, ddof=1))
    mp_se, mp_lo, mp_hi = ci95(mp, sp, len(pT))

    mD = float(np.mean(D_abs)); sD = float(np.std(D_abs, ddof=1))
    mD_se, mD_lo, mD_hi = ci95(mD, sD, len(D_abs))

    corr = float(np.corrcoef(pT, D_signed)[0, 1])

    rows.append({
        "rho": rho, "sigma_eff": seff, "C0": C0, "C1": C1,
        "E[p(T)]": mp, "SE[p(T)]": mp_se, "CI95_lo_pT": mp_lo, "CI95_hi_pT": mp_hi,
        "Var[p(T)]": float(sp*sp),
        "E[|D_s p(T)|]": mD, "SE[|D_s p(T)|]": mD_se, "CI95_lo_D": mD_lo, "CI95_hi_D": mD_hi,
        "Corr(D_s p(T), p(T))": corr, "M": M, "N": N,
        "T_days": T, "s_days": tgrid[s_index]
    })

    # ===== PANEL: Left = ensemble (mean ± SD + sample paths); Right = hexbin with colorbar =====
    fig = plt.figure(figsize=(10, 4.2))
    gs = GridSpec(1, 2, width_ratios=[1.1, 1.0], wspace=0.22, figure=fig)

    # Left panel: ensemble
    axL = fig.add_subplot(gs[0, 0])
    mean_path = p_paths.mean(axis=0)
    std_path = p_paths.std(axis=0, ddof=1)
    idx = rng.choice(M, size=min(PLOT_K, M), replace=False)
    for i in idx:
        axL.plot(tgrid, p_paths[i], lw=0.8, alpha=0.35)
    axL.plot(tgrid, mean_path, lw=2.0, label="Mean")
    axL.fill_between(tgrid, mean_path - std_path, mean_path + std_path, alpha=0.18, label="±1 SD")
    axL.set_xlabel("Time (days)")
    axL.set_ylabel("Prevalence $p(t)$")
    axL.set_title(f"Prevalence ensemble (ρ={rho:.1f}, σ_eff={seff:.3f})")
    axL.set_ylim(0, 1)
    axL.legend(frameon=False, ncol=2)

    # Right panel: hexbin with adjustable colorbar placed to the right
    axR = fig.add_subplot(gs[0, 1])
    hb = axR.hexbin(pT, D_abs, gridsize=45, bins='log', mincnt=1)
    axR.set_xlabel(r"$p(T)$")
    axR.set_ylabel(r"$|D_s p(T)|$ (with $s=T/2$)")
    axR.set_title("Joint density: $p(T)$ vs $|D_s p(T)|$")

    # Precise colorbar placement using axes_grid1
    divider = make_axes_locatable(axR)
    cax = divider.append_axes("right", size="5%", pad=0.10)  # customize size and pad here
    cb = fig.colorbar(hb, cax=cax)
    cb.set_label("log(count)")

    fig.tight_layout()
    for ext in ("png", "pdf", "svg"):
        fig.savefig(fig_dir / f"panel_rho_{rho:.1f}.{ext}")
    plt.close(fig)

# ---------- Save main summary table ----------
summary_df = pd.DataFrame(rows)
summary_path = tbl_dir / "IHM_MC_summary_with_CIs.csv"
summary_df.to_csv(summary_path, index=False)

# ---------- Convergence table by M for one rho (0.5) ----------
def one_run_summary(rho, M, N):
    p_paths, D_signed, seff, C0, C1, tgrid_, s_index_ = simulate_paths(
        rho, M, N, A, B, sigma_P, sigma_Q, p0, rng
    )
    pT = p_paths[:, -1]
    D_abs = np.abs(D_signed)
    mp = float(np.mean(pT)); sp = float(np.std(pT, ddof=1)); mp_se, mp_lo, mp_hi = ci95(mp, sp, len(pT))
    mD = float(np.mean(D_abs)); sD = float(np.std(D_abs, ddof=1)); mD_se, mD_lo, mD_hi = ci95(mD, sD, len(D_abs))
    corr = float(np.corrcoef(pT, D_signed)[0, 1])
    return {
        "rho": rho, "M": M, "N": N, "sigma_eff": float(sigma_eff(sigma_P, sigma_Q, rho)),
        "E[p(T)]": mp, "CI95_lo_pT": mp_lo, "CI95_hi_pT": mp_hi,
        "E[|D_s p(T)|]": mD, "CI95_lo_D": mD_lo, "CI95_hi_D": mD_hi,
        "Corr(D_s p(T), p(T))": corr
    }

rho_conv = 0.5
conv_rows = []
for Mtest in [2000, 5000, 10000]:
    conv_rows.append(one_run_summary(rho_conv, Mtest, N))
conv_df = pd.DataFrame(conv_rows)
conv_path = tbl_dir / "IHM_MC_convergence_by_M.csv"
conv_df.to_csv(conv_path, index=False)

# ---------- Step-size check: rho=0.5, fixed M=3000, compare N and 2N ----------
M_step = 3000
row_dt1 = one_run_summary(rho_conv, M_step, N)          # dt = T/N
row_dt2 = one_run_summary(rho_conv, M_step, 2*N)        # dt halved
row_dt1["dt_days"] = T/N
row_dt2["dt_days"] = T/(2*N)
step_df = pd.DataFrame([row_dt1, row_dt2])
step_path = tbl_dir / "IHM_MC_step_size_check.csv"
step_df.to_csv(step_path, index=False)

# ---------- Write LaTeX tables (compact booktabs) ----------
def write_latex_table(df, fname, caption, label):
    from io import StringIO
    buf = StringIO()
    # Build a compact LaTeX table
    buf.write("\\begin{table}[t]\n\\centering\n\\caption{" + caption + "}\n")
    buf.write("\\label{" + label + "}\n\\small\n\\begin{tabular}{")
    # Column alignment: center all
    buf.write("".join(["c" for _ in df.columns]))
    buf.write("}\n\\toprule\n")
    # Header
    buf.write(" & ".join([str(c).replace("_", "\\_") for c in df.columns]) + " \\\\\n\\midrule\n")
    # Rows
    for _, r in df.iterrows():
        vals = []
        for c in df.columns:
            v = r[c]
            if isinstance(v, float):
                vals.append(f"{v:.4f}")
            else:
                vals.append(str(v))
        buf.write(" & ".join(vals) + " \\\\\n")
    buf.write("\\bottomrule\n\\end{tabular}\n\\end{table}\n")
    text = buf.getvalue()
    path = tbl_dir / fname
    with open(path, "w") as f:
        f.write(text)
    return path

latex1 = write_latex_table(
    summary_df[["rho","sigma_eff","C0","C1","E[p(T)]","SE[p(T)]","CI95_lo_pT","CI95_hi_pT",
                "E[|D_s p(T)|]","SE[|D_s p(T)|]","CI95_lo_D","CI95_hi_D","Corr(D_s p(T), p(T))","M","N","T_days"]],
    "ihm_mc_summary_with_CIs_table.tex",
    "Monte Carlo summary across $\\rho$ with 95\\% CIs (affine contrast example; $T=60$ days).",
    "tab:ihm-mc-summary-CIs"
)

latex2 = write_latex_table(
    conv_df[["rho","M","N","sigma_eff","E[p(T)]","CI95_lo_pT","CI95_hi_pT",
             "E[|D_s p(T)|]","CI95_lo_D","CI95_hi_D","Corr(D_s p(T), p(T))"]],
    "ihm_convergence_by_M_table.tex",
    "Convergence check at $\\rho=0.5$ as $M$ increases (95\\% CIs).",
    "tab:ihm-convergence-M"
)

latex3 = write_latex_table(
    step_df[["rho","M","N","dt_days","E[p(T)]","E[|D_s p(T)|]","Corr(D_s p(T), p(T))"]],
    "ihm_step_size_check_table.tex",
    "Step-size check at $\\rho=0.5$ with $M=3000$: baseline $\\Delta t=T/N$ vs half-step.",
    "tab:ihm-stepsize"
)

print("WROTE TABLES:")
print(" -", summary_path.as_posix())
print(" -", conv_path.as_posix())
print(" -", step_path.as_posix())
print(" - LaTeX:", str(latex1), str(latex2), str(latex3))

print("PANEL FIGURES (per rho) saved to:", fig_dir.as_posix())


/tmp/ipython-input-277727892.py:169: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()
/tmp/ipython-input-277727892.py:169: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()
/tmp/ipython-input-277727892.py:169: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()
/tmp/ipython-input-277727892.py:169: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()


WROTE TABLES:
 - /content/final/ihm_outputs/tables/IHM_MC_summary_with_CIs.csv
 - /content/final/ihm_outputs/tables/IHM_MC_convergence_by_M.csv
 - /content/final/ihm_outputs/tables/IHM_MC_step_size_check.csv
 - LaTeX: /content/final/ihm_outputs/tables/ihm_mc_summary_with_CIs_table.tex /content/final/ihm_outputs/tables/ihm_convergence_by_M_table.tex /content/final/ihm_outputs/tables/ihm_step_size_check_table.tex
PANEL FIGURES (per rho) saved to: /content/final/ihm_outputs/figs


In [ ]:
# === IHM Monte Carlo (Affine + MB): summary + panels + adjustable colorbar ===
import os, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.gridspec import GridSpec
from mpl_toolkits.axes_grid1 import make_axes_locatable

# ===================== USER SETTINGS =====================
# Choose which model to generate: "affine" or "mb"
MODEL = "mb"   # <-- set to "affine" or "mb"

# Output folders
base_dir = Path("content/nonaffine/ihm_outputs_" + MODEL)
tbl_dir = base_dir / "tables"
fig_dir = base_dir / "figs"
tbl_dir.mkdir(parents=True, exist_ok=True)
fig_dir.mkdir(parents=True, exist_ok=True)

# Matplotlib (journal-ish)
plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 300,
    "font.size": 12,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "legend.fontsize": 11,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "axes.grid": True,
    "grid.alpha": 0.2,
    "grid.linestyle": "--",
    "axes.spines.top": False,
    "axes.spines.right": False
})

# Time scale
T = 60.0        # days
N = 300         # steps (dt ~ 0.2 day). For step-size test we also run 2N.
dt = T / N
tgrid = np.linspace(0.0, T, N+1)
s_index = N // 2
s_time = tgrid[s_index]

# Noise on P and Q + correlations to test
sigma_P = 0.20
sigma_Q = 0.15
rho_grid = [-0.5, 0.0, 0.5, 0.9]

# Monte Carlo size and #paths plotted on ensemble panel
M = 10000
PLOT_K = 50

# Initial prevalence
p0 = 0.10

# RNG
rng = np.random.default_rng(7)

# Colorbar placement (tweak as needed)
cbar_size = "5%"   # width
cbar_pad  = 0.10   # gap to the axes

# ========== Model parameters ==========
# Affine contrast: Δr(p) = A + B p
A = 0.05
B = 0.30

# MB parameters (vertical + horizontal transmission)
#   Δr_MB(p) = A_MB + B_MB * p/(1-p), with:
#   A_MB = r φ (τ-1) + λ + β N,   B_MB = - r φ (1-τ)
r = 0.10
phi = 0.9      # treat as exogenous on this horizon
tau = 0.8      # vertical transmission
lam = 0.00     # λ_P - λ_Q for the contrast
beta = 0.05    # horizontal conversion
N_pop = 1.0    # use a scale (e.g., normalized) to form A_MB

# ===================== Helpers =====================
def sigma_eff(sP, sQ, rho):
    return math.sqrt(sP**2 + sQ**2 - 2.0 * rho * sP * sQ)

def ci95(mean, sd, n):
    se = sd / np.sqrt(n)
    lo = mean - 1.96 * se
    hi = mean + 1.96 * se
    return se, lo, hi

# ----- Drift and derivative (affine) -----
def drift_affine(p, C0, C1):
    return p * (1.0 - p) * (C0 + C1 * p)

def dpb_affine(p, C0, C1, sigma_P, sigma_Q, rho):
    # ∂_p b(p) = (1-2p)*[Δr + noise] + p(1-p)*[∂_p Δr + ∂_p noise]
    # For affine Δr = A + B p, ∂_p Δr = B.
    xi = (A + B*p) + 2.0*(sigma_P**2*(1.0 - p) - sigma_Q**2*p) + 2.0*rho*sigma_P*sigma_Q*(2.0*p - 1.0)
    N1 = -2.0*sigma_P**2 - 2.0*sigma_Q**2 + 4.0*rho*sigma_P*sigma_Q
    return (1.0 - 2.0*p)*xi + (p*(1.0 - p))*(B + N1)

# ----- Drift and derivative (MB) -----
def L_MB(p, rho):
    # L_MB(p) = A_MB + N0 + N1 p + B_MB * p/(1-p)
    A_MB = r*phi*(tau - 1.0) + lam + beta * N_pop
    B_MB = - r*phi*(1.0 - tau)
    N0 =  2.0*sigma_P**2 - 2.0*rho*sigma_P*sigma_Q
    N1 = -2.0*sigma_P**2 - 2.0*sigma_Q**2 + 4.0*rho*sigma_P*sigma_Q
    return (A_MB + N0) + N1*p + B_MB * (p / (1.0 - p))

def drift_mb(p, rho):
    return p * (1.0 - p) * L_MB(p, rho)

def dpb_mb(p, rho):
    # ∂_p b_MB(p) = (1-2p) ξ_MB + p(1-p) [∂_p Δr_MB - 2σP^2 - 2σQ^2 + 4ρσPσQ]
    A_MB = r*phi*(tau - 1.0) + lam + beta * N_pop
    B_MB = - r*phi*(1.0 - tau)
    xi_mb = (A_MB + B_MB*(p/(1.0 - p))) + 2.0*(sigma_P**2*(1.0 - p) - sigma_Q**2*p) + 2.0*rho*sigma_P*sigma_Q*(2.0*p - 1.0)
    dDelta_mb = - r*phi*(1.0 - tau) * (1.0 - p)**(-2.0)
    N1 = -2.0*sigma_P**2 - 2.0*sigma_Q**2 + 4.0*rho*sigma_P*sigma_Q
    return (1.0 - 2.0*p)*xi_mb + p*(1.0 - p)*(dDelta_mb + N1)

# ----- Simulator -----
def simulate_paths(rho, M, N, model, p0, rng):
    seff = sigma_eff(sigma_P, sigma_Q, rho)
    dt = T / N
    tgrid = np.linspace(0.0, T, N+1)
    s_index = N // 2

    # affine precompute
    C0 = A + 2.0*sigma_P**2 - 2.0*rho*sigma_P*sigma_Q
    C1 = B - 2.0*sigma_P**2 - 2.0*sigma_Q**2 + 4.0*rho*sigma_P*sigma_Q

    p = np.empty((M, N+1), dtype=float)
    p[:, 0] = p0
    dW = rng.normal(0.0, math.sqrt(dt), size=(M, N))
    eps = 1e-6

    for k in range(N):
        pk = p[:, k]
        if model == "affine":
            drift = drift_affine(pk, C0, C1)
        elif model == "mb":
            drift = drift_mb(pk, rho)
        else:
            raise ValueError("MODEL must be 'affine' or 'mb'")
        diff = seff * pk * (1.0 - pk)
        p[:, k+1] = np.clip(pk + drift * dt + diff * dW[:, k], eps, 1.0 - eps)

    # Malliavin D_s p(T) via tangent exponential (s = T/2)
    pk = p[:, s_index:-1]
    dW_seg = dW[:, s_index:]
    if model == "affine":
        dp_b = dpb_affine(pk, C0, C1, sigma_P, sigma_Q, rho)
    else:
        dp_b = dpb_mb(pk, rho)

    term1 = np.sum(dp_b * dt - 0.5*(seff**2)*(1.0 - 2.0*pk)**2 * dt, axis=1)
    term2 = np.sum(seff * (1.0 - 2.0*pk) * dW_seg, axis=1)
    D_signed = seff * p[:, s_index] * (1.0 - p[:, s_index]) * np.exp(term1 + term2)

    return p, D_signed, seff, C0, C1, tgrid, s_index

# ===================== MAIN per-ρ run =====================
rows = []
for rho in rho_grid:
    p_paths, D_signed, seff, C0, C1, tgrid, s_index = simulate_paths(
        rho, M, N, MODEL, p0, rng
    )
    pT = p_paths[:, -1]
    D_abs = np.abs(D_signed)

    # stats + CIs
    mp = float(np.mean(pT)); sp = float(np.std(pT, ddof=1))
    mp_se, mp_lo, mp_hi = ci95(mp, sp, len(pT))
    mD = float(np.mean(D_abs)); sD = float(np.std(D_abs, ddof=1))
    mD_se, mD_lo, mD_hi = ci95(mD, sD, len(D_abs))
    corr = float(np.corrcoef(pT, D_signed)[0, 1])

    rows.append({
        "model": MODEL, "rho": rho, "sigma_eff": seff,
        "C0": C0 if MODEL=="affine" else np.nan,
        "C1": C1 if MODEL=="affine" else np.nan,
        "E[p(T)]": mp, "SE[p(T)]": mp_se, "CI95_lo_pT": mp_lo, "CI95_hi_pT": mp_hi,
        "Var[p(T)]": float(sp*sp),
        "E[|D_s p(T)|]": mD, "SE[|D_s p(T)|]": mD_se, "CI95_lo_D": mD_lo, "CI95_hi_D": mD_hi,
        "Corr(D_s p(T), p(T))": corr, "M": M, "N": N, "T_days": T, "s_days": tgrid[s_index]
    })

    # ===== PANEL: left ensemble, right hexbin + colorbar =====
    fig = plt.figure(figsize=(10, 4.2))
    gs = GridSpec(1, 2, width_ratios=[1.1, 1.0], wspace=0.22, figure=fig)

    # Left: ensemble
    axL = fig.add_subplot(gs[0, 0])
    mean_path = p_paths.mean(axis=0)
    std_path = p_paths.std(axis=0, ddof=1)
    idx = rng.choice(M, size=min(PLOT_K, M), replace=False)
    for i in idx:
        axL.plot(tgrid, p_paths[i], lw=0.8, alpha=0.35)
    title_left = (f"Prevalence ensemble (ρ={rho:.1f}, σ_eff={seff:.3f})"
                  if MODEL=="affine"
                  else f"MB ensemble (ρ={rho:.1f}, σ_eff={seff:.3f}, τ={tau:.2f}, β={beta:.2f})")
    axL.plot(tgrid, mean_path, lw=2.0, label="Mean")
    axL.fill_between(tgrid, mean_path - std_path, mean_path + std_path, alpha=0.18, label="±1 SD")
    axL.set_xlabel("Time (days)")
    axL.set_ylabel("Prevalence $p(t)$")
    axL.set_title(title_left)
    axL.set_ylim(0, 1)
    axL.legend(frameon=False, ncol=2)

    # Right: hexbin + colorbar
    axR = fig.add_subplot(gs[0, 1])
    hb = axR.hexbin(pT, D_abs, gridsize=45, bins='log', mincnt=1)
    axR.set_xlabel(r"$p(T)$")
    axR.set_ylabel(r"$|D_s p(T)|$ (with $s=T/2$)")
    axR.set_title("Joint density: $p(T)$ vs $|D_s p(T)|$")
    divider = make_axes_locatable(axR)
    cax = divider.append_axes("right", size=cbar_size, pad=cbar_pad)
    cb = fig.colorbar(hb, cax=cax)
    cb.set_label("log(count)")

    fig.tight_layout()
    base = f"panel_{MODEL}_rho_{rho:.1f}"
    for ext in ("png", "pdf", "svg"):
        fig.savefig(fig_dir / f"{base}.{ext}")
    plt.close(fig)

# ----- Save summary -----
summary_df = pd.DataFrame(rows)
summary_path = tbl_dir / f"{MODEL.upper()}_MC_summary_with_CIs.csv"
summary_df.to_csv(summary_path, index=False)

# ----- Convergence in M for one rho -----
def one_run_summary(rho, M, N, model):
    p_paths, D_signed, seff, C0, C1, tgrid_, s_index_ = simulate_paths(
        rho, M, N, model, p0, rng
    )
    pT = p_paths[:, -1]
    D_abs = np.abs(D_signed)
    mp = float(np.mean(pT)); sp = float(np.std(pT, ddof=1))
    mp_se, mp_lo, mp_hi = ci95(mp, sp, len(pT))
    mD = float(np.mean(D_abs)); sD = float(np.std(D_abs, ddof=1))
    mD_se, mD_lo, mD_hi = ci95(mD, sD, len(D_abs))
    corr = float(np.corrcoef(pT, D_signed)[0, 1])
    return {
        "model": model, "rho": rho, "M": M, "N": N, "sigma_eff": float(sigma_eff(sigma_P, sigma_Q, rho)),
        "E[p(T)]": mp, "CI95_lo_pT": mp_lo, "CI95_hi_pT": mp_hi,
        "E[|D_s p(T)|]": mD, "CI95_lo_D": mD_lo, "CI95_hi_D": mD_hi,
        "Corr(D_s p(T), p(T))": corr
    }

rho_conv = 0.5
conv_rows = []
for Mtest in [2000, 5000, 10000]:
    conv_rows.append(one_run_summary(rho_conv, Mtest, N, MODEL))
conv_df = pd.DataFrame(conv_rows)
conv_path = tbl_dir / f"{MODEL.upper()}_MC_convergence_by_M.csv"
conv_df.to_csv(conv_path, index=False)

# ----- Step-size check (N vs 2N) -----
M_step = 3000
row_dt1 = one_run_summary(rho_conv, M_step, N, MODEL)
row_dt2 = one_run_summary(rho_conv, M_step, 2*N, MODEL)
row_dt1["dt_days"] = T/N
row_dt2["dt_days"] = T/(2*N)
step_df = pd.DataFrame([row_dt1, row_dt2])
step_path = tbl_dir / f"{MODEL.upper()}_MC_step_size_check.csv"
step_df.to_csv(step_path, index=False)

# ----- Compact LaTeX tables -----
def write_latex_table(df, fname, caption, label):
    from io import StringIO
    buf = StringIO()
    buf.write("\\begin{table}[t]\n\\centering\n\\caption{" + caption + "}\n")
    buf.write("\\label{" + label + "}\n\\small\n\\begin{tabular}{")
    buf.write("".join(["c" for _ in df.columns]))
    buf.write("}\n\\toprule\n")
    buf.write(" & ".join([str(c).replace("_","\\_") for c in df.columns]) + " \\\\\n\\midrule\n")
    for _, r in df.iterrows():
        vals=[]
        for c in df.columns:
            v = r[c]
            if isinstance(v, float):
                vals.append(f"{v:.4f}")
            else:
                vals.append(str(v))
        buf.write(" & ".join(vals) + " \\\\\n")
    buf.write("\\bottomrule\n\\end{tabular}\n\\end{table}\n")
    path = tbl_dir / fname
    with open(path, "w") as f:
        f.write(buf.getvalue())
    return path

cols_sum = ["model","rho","sigma_eff","E[p(T)]","SE[p(T)]","CI95_lo_pT","CI95_hi_pT",
            "E[|D_s p(T)|]","SE[|D_s p(T)|]","CI95_lo_D","CI95_hi_D","Corr(D_s p(T), p(T))","M","N","T_days"]
if MODEL == "affine":
    cols_sum = ["model","rho","sigma_eff","C0","C1"] + cols_sum[3:]

latex1 = write_latex_table(
    summary_df[cols_sum],
    f"{MODEL}_mc_summary_with_CIs_table.tex",
    f"Monte Carlo summary across $\\rho$ with 95\\% CIs ({MODEL} example; $T=60$ days).",
    f"tab:{MODEL}-mc-summary-CIs"
)

latex2 = write_latex_table(
    conv_df[["model","rho","M","N","sigma_eff","E[p(T)]","CI95_lo_pT","CI95_hi_pT",
             "E[|D_s p(T)|]","CI95_lo_D","CI95_hi_D","Corr(D_s p(T), p(T))"]],
    f"{MODEL}_convergence_by_M_table.tex",
    "Convergence check at $\\rho=0.5$ as $M$ increases (95\\% CIs).",
    f"tab:{MODEL}-convergence-M"
)

latex3 = write_latex_table(
    step_df[["model","rho","M","N","dt_days","E[p(T)]","E[|D_s p(T)|]","Corr(D_s p(T), p(T))"]],
    f"{MODEL}_step_size_check_table.tex",
    "Step-size check at $\\rho=0.5$ with $M=3000$: baseline $\\Delta t=T/N$ vs half-step.",
    f"tab:{MODEL}-stepsize"
)

print("WROTE TABLES:")
print(" -", summary_path.as_posix())
print(" -", conv_path.as_posix())
print(" -", step_path.as_posix())
print(" - LaTeX:", str(latex1), str(latex2), str(latex3))
print("PANEL FIGURES saved to:", fig_dir.as_posix())


/tmp/ipython-input-502804191.py:222: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()
/tmp/ipython-input-502804191.py:222: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()
/tmp/ipython-input-502804191.py:222: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()
/tmp/ipython-input-502804191.py:222: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()


WROTE TABLES:
 - content/nonaffine/ihm_outputs_mb/tables/MB_MC_summary_with_CIs.csv
 - content/nonaffine/ihm_outputs_mb/tables/MB_MC_convergence_by_M.csv
 - content/nonaffine/ihm_outputs_mb/tables/MB_MC_step_size_check.csv
 - LaTeX: content/nonaffine/ihm_outputs_mb/tables/mb_mc_summary_with_CIs_table.tex content/nonaffine/ihm_outputs_mb/tables/mb_convergence_by_M_table.tex content/nonaffine/ihm_outputs_mb/tables/mb_step_size_check_table.tex
PANEL FIGURES saved to: content/nonaffine/ihm_outputs_mb/figs


In [ ]:
# === MB Heatmaps across (tau, beta) for different rho values ===
# Produces:
#   - CSVs: ihm_outputs_mb_sweeps/tables/MB_heatmap_grid_rho_*.csv
#   - Figures: ihm_outputs_mb_sweeps/figs/mb_heatmaps_{neg,pos}.{png,pdf,svg}

import math, numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
from mpl_toolkits.axes_grid1 import make_axes_locatable

# ---------------- Folders ----------------
base_dir = Path("content/ihm_outputs_mb_sweeps")
tbl_dir = base_dir / "tables"
fig_dir = base_dir / "figs"
tbl_dir.mkdir(parents=True, exist_ok=True)
fig_dir.mkdir(parents=True, exist_ok=True)

# ---------------- Style ----------------
plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 300,
    "font.size": 12,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "legend.fontsize": 11,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "axes.grid": True,
    "grid.alpha": 0.15,
    "grid.linestyle": "--",
    "axes.spines.top": False,
    "axes.spines.right": False
})

# ---------------- Simulation settings ----------------
T = 60.0         # days
N = 300          # steps => dt ~ 0.2 day
dt = T / N
tgrid = np.linspace(0.0, T, N+1)
s_index = N // 2

# Noise parameters
sigma_P = 0.20
sigma_Q = 0.15

# MB biological parameters (defaults; tau, beta will be swept)
r = 0.10
phi = 0.90
lam = 0.00    # lambda_P - lambda_Q
N_pop = 1.0   # scale for beta * N

# Sweep grids (τ in [50,80]%; β in [0,30]%)
tau_grid = np.array([0.50, 0.60, 0.70, 0.80])
beta_grid = np.array([0.00, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30])
rho_list  = [-0.5, 0.0, 0.5, 0.9]

# Monte Carlo
M_heat = 10000
p0 = 0.10
rng = np.random.default_rng(12)

# ---------------- MB drift pieces ----------------
def sigma_eff(sP, sQ, rho):
    return math.sqrt(sP**2 + sQ**2 - 2.0 * rho * sP * sQ)

def A_MB_val(tau, beta):
    return r*phi*(tau - 1.0) + lam + beta * N_pop

def B_MB_val(tau):
    return - r*phi*(1.0 - tau)

def L_MB(p, rho, tau, beta):
    # L(p) = Δr_MB(p) + σ_P^2(1-p) - σ_Q^2 p + ρ σ_P σ_Q (2p - 1)
    A_mb = A_MB_val(tau, beta)
    B_mb = B_MB_val(tau)
    return (A_mb + B_mb * (p / (1.0 - p))
            + sigma_P**2 * (1.0 - p) - sigma_Q**2 * p + rho * sigma_P * sigma_Q * (2.0*p - 1.0))

def dDelta_mb_dp(p, tau):
    # ∂_p Δr_MB(p) = - r φ (1-τ) (1-p)^{-2}
    return - r*phi*(1.0 - tau) * (1.0 - p)**(-2.0)

def dpb_mb(p, rho, tau, beta):
    # ∂_p b = (1-2p) L(p) + p(1-p) * [ ∂_p Δr_MB + ( -σ_P^2 - σ_Q^2 + 2 ρ σ_P σ_Q ) ]
    L = L_MB(p, rho, tau, beta)
    N1 = - sigma_P**2 - sigma_Q**2 + 2.0 * rho * sigma_P * sigma_Q
    return (1.0 - 2.0*p) * L + p * (1.0 - p) * (dDelta_mb_dp(p, tau) + N1)

# ---------------- Simulator (MB) ----------------
def simulate_mb_once(rho, tau, beta):
    seff = sigma_eff(sigma_P, sigma_Q, rho)
    p = np.empty((M_heat, N+1), dtype=float)
    p[:, 0] = p0
    dW = rng.normal(0.0, math.sqrt(dt), size=(M_heat, N))
    eps = 1e-6
    for k in range(N):
        pk = p[:, k]
        drift = pk * (1.0 - pk) * L_MB(pk, rho, tau, beta)
        diff  = seff * pk * (1.0 - pk)
        p[:, k+1] = np.clip(pk + drift * dt + diff * dW[:, k], eps, 1.0 - eps)

    # Malliavin tangent (optional; not used for the prevalence heatmaps)
    pk_seg = p[:, s_index:-1]
    dW_seg = dW[:, s_index:]
    dp_b   = dpb_mb(pk_seg, rho, tau, beta)
    term1  = np.sum(dp_b * dt - 0.5*(seff**2)*(1.0 - 2.0*pk_seg)**2 * dt, axis=1)
    term2  = np.sum(seff * (1.0 - 2.0*pk_seg) * dW_seg, axis=1)
    D_s    = seff * p[:, s_index] * (1.0 - p[:, s_index]) * np.exp(term1 + term2)

    return p[:, -1], D_s

# ---------------- Sweep and collect ----------------
all_results = {}
all_means = []

for rho in rho_list:
    rows = []
    for tau in tau_grid:
        for beta in beta_grid:
            pT, D_s = simulate_mb_once(rho, tau, beta)
            mean_pT = float(np.mean(pT))
            rows.append({
                "rho": rho, "tau": tau, "beta": beta,
                "E_pT": mean_pT,              # fraction
                "E_pT_pct": 100.0 * mean_pT,  # percent
                "E_absD": float(np.mean(np.abs(D_s)))
            })
            all_means.append(100.0 * mean_pT)
    df = pd.DataFrame(rows)
    all_results[rho] = df
    df.to_csv(tbl_dir / f"MB_heatmap_grid_rho_{rho:+.1f}.csv", index=False)

# Use a common color scale across rhos (percent)
vmin = float(np.floor(min(all_means)))
vmax = float(np.ceil(max(all_means)))

# ---------------- Plot heatmaps ----------------
def plot_two_heatmaps(rho_a, rho_b, fname_base):
    fig, axes = plt.subplots(1, 2, figsize=(11.2, 4.6))
    for ax, rho in zip(axes, [rho_a, rho_b]):
        df = all_results[rho]
        pivot = df.pivot(index="tau", columns="beta", values="E_pT_pct")
        pivot = pivot.loc[tau_grid, beta_grid]
        im = ax.imshow(pivot.values, origin="lower", aspect="auto",
                       extent=[100*beta_grid[0], 100*beta_grid[-1], 100*tau_grid[0], 100*tau_grid[-1]],
                       vmin=vmin, vmax=vmax)
        ax.set_title(f"MB: mean prevalence (%)  —  $\\rho={rho:+.1f}$")
        ax.set_xlabel("Horizontal transmission $\\beta$ (%)")
        ax.set_ylabel("Vertical transmission $\\tau$ (%)")
        ax.set_xticks(100*beta_grid)
        ax.set_yticks(100*tau_grid)
        ax.set_xlim(100*beta_grid[0], 100*beta_grid[-1])
        ax.set_ylim(100*tau_grid[0], 100*tau_grid[-1])
        divider = make_axes_locatable(ax)
        cax = divider.append_axes("right", size="4.5%", pad=0.10)
        cb = fig.colorbar(im, cax=cax)
        cb.set_label("Mean $p(T)$ (%)")
    fig.tight_layout()
    for ext in ("png", "pdf", "svg"):
        fig.savefig(fig_dir / f"{fname_base}.{ext}")
    plt.close(fig)

# Negative/zero and positive pairs
plot_two_heatmaps(-0.5, 0.0, "mb_heatmaps_neg")
plot_two_heatmaps(0.5, 0.9, "mb_heatmaps_pos")

print("Saved CSV grids to:", tbl_dir.as_posix())
print("Saved heatmap figures to:", fig_dir.as_posix())


Saved CSV grids to: content/ihm_outputs_mb_sweeps/tables
Saved heatmap figures to: content/ihm_outputs_mb_sweeps/figs


In [ ]:
# === MB Heatmaps across (tau, beta) for rho in {-0.5, 0, 0.5, 0.9} WITH ANNOTATIONS ===
# Writes:
#   tables/MB_heatmap_grid_rho_*.csv (per-ρ grid of means)
#   figs/mb_heatmaps_neg.{png,pdf,svg}   ->  (ρ=-0.5, 0.0), annotated mean prevalence (%)
#   figs/mb_heatmaps_pos.{png,pdf,svg}   ->  (ρ=+0.5, +0.9), annotated mean prevalence (%)
#   figs/mb_heatmaps_neg_D.{png,pdf,svg} ->  (ρ=-0.5, 0.0), annotated mean |D_s p(T)|  [optional]
#   figs/mb_heatmaps_pos_D.{png,pdf,svg} ->  (ρ=+0.5, +0.9), annotated mean |D_s p(T)|  [optional]

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from mpl_toolkits.axes_grid1 import make_axes_locatable

# ---------------- Folders ----------------
base_dir = Path("HEATMAPS/ihm_outputs_mb_sweeps")
tbl_dir = base_dir / "tables"
fig_dir = base_dir / "figs"
tbl_dir.mkdir(parents=True, exist_ok=True)
fig_dir.mkdir(parents=True, exist_ok=True)

# ---------------- Style ----------------
plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 300,
    "font.size": 12,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "legend.fontsize": 11,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "axes.grid": True,
    "grid.alpha": 0.15,
    "grid.linestyle": "--",
    "axes.spines.top": False,
    "axes.spines.right": False
})

# ---------------- Simulation settings ----------------
T = 60.0         # days
N = 300          # steps => dt ~ 0.2 day
dt = T / N
tgrid = np.linspace(0.0, T, N+1)
s_index = N // 2

# Noise parameters
sigma_P = 0.20
sigma_Q = 0.15

# MB biological parameters (defaults; tau, beta will be swept)
r = 0.10
phi = 0.90
lam = 0.00    # lambda_P - lambda_Q
N_pop = 1.0   # scale for beta * N

# Sweep grids (τ in [50,80]%; β in [0,30]% in 5% steps)
tau_grid  = np.array([0.50, 0.60, 0.70, 0.80])
beta_grid = np.array([0.00, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30])
rho_list  = [-0.5, 0.0, 0.5, 0.9]

# Monte Carlo
M_heat = 10_000
p0 = 0.10
rng = np.random.default_rng(12)

# ---------------- MB drift pieces ----------------
def sigma_eff(sP, sQ, rho):
    return math.sqrt(sP**2 + sQ**2 - 2.0 * rho * sP * sQ)

def A_MB_val(tau, beta):
    return r*phi*(tau - 1.0) + lam + beta * N_pop

def B_MB_val(tau):
    return - r*phi*(1.0 - tau)

def L_MB(p, rho, tau, beta):
    # L(p) = Δr_MB(p) + σ_P^2(1-p) - σ_Q^2 p + ρ σ_P σ_Q (2p - 1)
    A_mb = A_MB_val(tau, beta)
    B_mb = B_MB_val(tau)
    return (A_mb + B_mb * (p / (1.0 - p))
            + sigma_P**2 * (1.0 - p) - sigma_Q**2 * p + rho * sigma_P * sigma_Q * (2.0*p - 1.0))

def dDelta_mb_dp(p, tau):
    # ∂_p Δr_MB(p) = - r φ (1-τ) (1-p)^{-2}
    return - r*phi*(1.0 - tau) * (1.0 - p)**(-2.0)

def dpb_mb(p, rho, tau, beta):
    # ∂_p b = (1-2p) L(p) + p(1-p) * [ ∂_p Δr_MB + ( -σ_P^2 - σ_Q^2 + 2 ρ σ_P σ_Q ) ]
    L = L_MB(p, rho, tau, beta)
    N1 = - sigma_P**2 - sigma_Q**2 + 2.0 * rho * sigma_P * sigma_Q
    return (1.0 - 2.0*p) * L + p * (1.0 - p) * (dDelta_mb_dp(p, tau) + N1)

# ---------------- Simulator (MB) ----------------
def simulate_mb_once(rho, tau, beta):
    seff = sigma_eff(sigma_P, sigma_Q, rho)
    p = np.empty((M_heat, N+1), dtype=float)
    p[:, 0] = p0
    dW = rng.normal(0.0, math.sqrt(dt), size=(M_heat, N))
    eps = 1e-6
    for k in range(N):
        pk = p[:, k]
        drift = pk * (1.0 - pk) * L_MB(pk, rho, tau, beta)
        diff  = seff * pk * (1.0 - pk)
        p[:, k+1] = np.clip(pk + drift * dt + diff * dW[:, k], eps, 1.0 - eps)

    # Malliavin tangent (s = T/2) — we can compute |D_s p(T)| also for a heatmap
    pk_seg = p[:, s_index:-1]
    dW_seg = dW[:, s_index:]
    dp_b   = dpb_mb(pk_seg, rho, tau, beta)
    term1  = np.sum(dp_b * dt - 0.5*(seff**2)*(1.0 - 2.0*pk_seg)**2 * dt, axis=1)
    term2  = np.sum(seff * (1.0 - 2.0*pk_seg) * dW_seg, axis=1)
    D_s    = seff * p[:, s_index] * (1.0 - p[:, s_index]) * np.exp(term1 + term2)

    return p[:, -1], D_s

# ---------------- Sweep and collect ----------------
all_results = {}
all_means_pct = []
all_means_D   = []

for rho in rho_list:
    rows = []
    for tau in tau_grid:
        for beta in beta_grid:
            pT, D_s = simulate_mb_once(rho, tau, beta)
            mean_pT = float(np.mean(pT))
            mean_D  = float(np.mean(np.abs(D_s)))
            rows.append({
                "rho": rho, "tau": tau, "beta": beta,
                "E_pT": mean_pT,              # fraction
                "E_pT_pct": 100.0 * mean_pT,  # percent
                "E_absD": mean_D
            })
            all_means_pct.append(100.0 * mean_pT)
            all_means_D.append(mean_D)
    df = pd.DataFrame(rows)
    all_results[rho] = df
    df.to_csv(tbl_dir / f"MB_heatmap_grid_rho_{rho:+.1f}.csv", index=False)

# Common color scales across rhos
vmin_pct = float(np.floor(min(all_means_pct)))
vmax_pct = float(np.ceil(max(all_means_pct)))
vmin_D   = float(np.floor(1000*min(all_means_D))/1000.0)   # round to 0.001
vmax_D   = float(np.ceil (1000*max(all_means_D))/1000.0)

# ---------------- Plot heatmaps WITH annotated values ----------------
def _bin_edges_from_centers(centers):
    """Compute edges from ordered bin centers."""
    centers = np.asarray(centers, dtype=float)
    edges = np.empty(len(centers) + 1, dtype=float)
    if len(centers) > 1:
        edges[1:-1] = 0.5 * (centers[:-1] + centers[1:])
        edges[0]  = centers[0]  - 0.5 * (centers[1] - centers[0])
        edges[-1] = centers[-1] + 0.5 * (centers[-1] - centers[-2])
    else:
        edges[:] = [centers[0] - 0.5, centers[0] + 0.5]
    return edges

def _annotate_cells(ax, X_centers, Y_centers, M, fmt, vmin, vmax):
    threshold = 0.5 * (vmin + vmax)
    for iy, y in enumerate(Y_centers):
        for ix, x in enumerate(X_centers):
            val = M[iy, ix]
            color = "white" if val >= threshold else "black"
            ax.text(x, y, fmt.format(val), ha="center", va="center", color=color, fontsize=13)

def plot_pair_annotated(rho_a, rho_b, fname_base, which="pct"):
    """
    Make a 2-panel figure of annotated heatmaps.
    which = "pct" -> mean prevalence (%) with integer labels
            "D"   -> mean |D_s p(T)| with 3 decimals
    """
    fig, axes = plt.subplots(1, 2, figsize=(11.6, 4.8))

    # Percent axes
    X_centers = 100.0 * beta_grid  # columns
    Y_centers = 100.0 * tau_grid   # rows
    X_edges   = _bin_edges_from_centers(X_centers)
    Y_edges   = _bin_edges_from_centers(Y_centers)

    for ax, rho in zip(axes, [rho_a, rho_b]):
        df = all_results[rho]
        if which == "pct":
            vals = df.pivot(index="tau", columns="beta", values="E_pT_pct").loc[tau_grid, beta_grid].values
            vmin, vmax = vmin_pct, vmax_pct
            fmt = "{:.0f}"    # integer %
            cbar_label = "Mean $p(T)$ (%)"
            title = f"MB: mean prevalence (%) — $\\rho={rho:+.1f}$"
        else:
            vals = df.pivot(index="tau", columns="beta", values="E_absD").loc[tau_grid, beta_grid].values
            vmin, vmax = vmin_D, vmax_D
            fmt = "{:.3f}"
            cbar_label = r"Mean $|D_s p(T)|$"
            title = f"MB: mean $|D_s p(T)|$ — $\\rho={rho:+.1f}$"

        im = ax.imshow(
            vals, origin="lower", aspect="auto",
            extent=[X_edges[0], X_edges[-1], Y_edges[0], Y_edges[-1]],
            vmin=vmin, vmax=vmax
        )
        ax.set_title(title)
        ax.set_xlabel("Horizontal transmission $\\beta$ (%)")
        ax.set_ylabel("Vertical transmission $\\tau$ (%)")
        ax.set_xticks(X_centers)
        ax.set_yticks(Y_centers)
        ax.set_xlim(X_edges[0], X_edges[-1])
        ax.set_ylim(Y_edges[0], Y_edges[-1])

        # Colorbar
        divider = make_axes_locatable(ax)
        cax = divider.append_axes("right", size="4.5%", pad=0.10)
        cb = fig.colorbar(im, cax=cax)
        cb.set_label(cbar_label)

        # Annotate
        _annotate_cells(ax, X_centers, Y_centers, vals, fmt, vmin, vmax)

    fig.tight_layout()
    for ext in ("png", "pdf", "svg"):
        fig.savefig(fig_dir / f"{fname_base}.{ext}")
    plt.close(fig)

# Negative/zero and positive pairs — prevalence (%)
plot_pair_annotated(-0.5, 0.0, "mb_heatmaps_neg", which="pct")
plot_pair_annotated( 0.5, 0.9, "mb_heatmaps_pos", which="pct")

# Optional: the same for mean |D_s p(T)| (comment out if not needed)
plot_pair_annotated(-0.5, 0.0, "mb_heatmaps_neg_D", which="D")
plot_pair_annotated( 0.5, 0.9, "mb_heatmaps_pos_D", which="D")

print("Saved CSV grids to:", tbl_dir.as_posix())
print("Saved heatmap figures to:", fig_dir.as_posix())


Saved CSV grids to: HEATMAPS/ihm_outputs_mb_sweeps/tables
Saved heatmap figures to: HEATMAPS/ihm_outputs_mb_sweeps/figs


In [1]:
# === MB Heatmaps across (tau, beta) with ANNOTATED CELLS ===
# Prevalence labels = integer %.
# |D_s p(T)| labels = integer % (by default), or 2 decimals if ANNOTATE_D_AS_PERCENT=False.

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from mpl_toolkits.axes_grid1 import make_axes_locatable

# ---------------- User switches ----------------
ANNOTATE_D_AS_PERCENT = True   # if True: show |D_s p(T)| in %, integer labels; else raw, 2 decimals

# ---------------- Folders ----------------
base_dir = Path("heatMAPS/ihm_outputs_mb_sweeps")
tbl_dir = base_dir / "tables"
fig_dir = base_dir / "figs"
tbl_dir.mkdir(parents=True, exist_ok=True)
fig_dir.mkdir(parents=True, exist_ok=True)

# ---------------- Style ----------------
plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 300,
    "font.size": 12,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "legend.fontsize": 11,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "axes.grid": True,
    "grid.alpha": 0.15,
    "grid.linestyle": "--",
    "axes.spines.top": False,
    "axes.spines.right": False
})

# ---------------- Simulation settings ----------------
T = 60.0         # days
N = 300          # steps => dt ~ 0.2 day
dt = T / N
tgrid = np.linspace(0.0, T, N+1)
s_index = N // 2

# Noise parameters
sigma_P = 0.20
sigma_Q = 0.15

# MB biological parameters (defaults; tau, beta will be swept)
r = 0.10
phi = 0.90
lam = 0.00    # lambda_P - lambda_Q
N_pop = 1.0   # scale for beta * N

# Sweep grids (τ in [50,80]%; β in [0,30]% in 5% steps)
tau_grid  = np.array([0.50, 0.60, 0.70, 0.80])
beta_grid = np.array([0.00, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30])
rho_list  = [-0.5, 0.0, 0.5, 0.9]

# Monte Carlo
M_heat = 10_000
p0 = 0.10
rng = np.random.default_rng(12)

# ---------------- MB drift pieces ----------------
def sigma_eff(sP, sQ, rho):
    return math.sqrt(sP**2 + sQ**2 - 2.0 * rho * sP * sQ)

def A_MB_val(tau, beta):
    return r*phi*(tau - 1.0) + lam + beta * N_pop

def B_MB_val(tau):
    return - r*phi*(1.0 - tau)

def L_MB(p, rho, tau, beta):
    # L(p) = Δr_MB(p) + σ_P^2(1-p) - σ_Q^2 p + ρ σ_P σ_Q (2p - 1)
    A_mb = A_MB_val(tau, beta)
    B_mb = B_MB_val(tau)
    return (A_mb + B_mb * (p / (1.0 - p))
            + sigma_P**2 * (1.0 - p) - sigma_Q**2 * p + rho * sigma_P * sigma_Q * (2.0*p - 1.0))

def dDelta_mb_dp(p, tau):
    # ∂_p Δr_MB(p) = - r φ (1-τ) (1-p)^{-2}
    return - r*phi*(1.0 - tau) * (1.0 - p)**(-2.0)

def dpb_mb(p, rho, tau, beta):
    # ∂_p b = (1-2p) L(p) + p(1-p) * [ ∂_p Δr_MB + ( -σ_P^2 - σ_Q^2 + 2 ρ σ_P σ_Q ) ]
    L = L_MB(p, rho, tau, beta)
    N1 = - sigma_P**2 - sigma_Q**2 + 2.0 * rho * sigma_P * sigma_Q
    return (1.0 - 2.0*p) * L + p * (1.0 - p) * (dDelta_mb_dp(p, tau) + N1)

# ---------------- Simulator (MB) ----------------
def simulate_mb_once(rho, tau, beta):
    seff = sigma_eff(sigma_P, sigma_Q, rho)
    p = np.empty((M_heat, N+1), dtype=float)
    p[:, 0] = p0
    dW = rng.normal(0.0, math.sqrt(dt), size=(M_heat, N))
    eps = 1e-6
    for k in range(N):
        pk = p[:, k]
        drift = pk * (1.0 - pk) * L_MB(pk, rho, tau, beta)
        diff  = seff * pk * (1.0 - pk)
        p[:, k+1] = np.clip(pk + drift * dt + diff * dW[:, k], eps, 1.0 - eps)

    # Malliavin tangent (s = T/2)
    pk_seg = p[:, s_index:-1]
    dW_seg = dW[:, s_index:]
    dp_b   = dpb_mb(pk_seg, rho, tau, beta)
    term1  = np.sum(dp_b * dt - 0.5*(seff**2)*(1.0 - 2.0*pk_seg)**2 * dt, axis=1)
    term2  = np.sum(seff * (1.0 - 2.0*pk_seg) * dW_seg, axis=1)
    D_s    = seff * p[:, s_index] * (1.0 - p[:, s_index]) * np.exp(term1 + term2)

    return p[:, -1], D_s

# ---------------- Sweep and collect ----------------
all_results = {}
all_means_pct = []
all_means_D_plot = []  # store in the same units we will PLOT/ANNOTATE

for rho in rho_list:
    rows = []
    for tau in tau_grid:
        for beta in beta_grid:
            pT, D_s = simulate_mb_once(rho, tau, beta)
            mean_pT = float(np.mean(pT))
            mean_D  = float(np.mean(np.abs(D_s)))

            # For plotting/annotation: choose units for |D| based on switch
            mean_D_plot = 100.0 * mean_D if ANNOTATE_D_AS_PERCENT else mean_D

            rows.append({
                "rho": rho, "tau": tau, "beta": beta,
                "E_pT": mean_pT,
                "E_pT_pct": 100.0 * mean_pT,          # prevalence in percent
                "E_absD": mean_D,                     # raw |D|
                "E_absD_plot": mean_D_plot            # |D| in the plotted units
            })
            all_means_pct.append(100.0 * mean_pT)
            all_means_D_plot.append(mean_D_plot)

    df = pd.DataFrame(rows)
    all_results[rho] = df
    df.to_csv(tbl_dir / f"MB_heatmap_grid_rho_{rho:+.1f}.csv", index=False)

# Common color scales across rhos
vmin_pct = float(np.floor(min(all_means_pct)))
vmax_pct = float(np.ceil(max(all_means_pct)))
vmin_Dp  = float(np.floor(min(all_means_D_plot)))
vmax_Dp  = float(np.ceil (max(all_means_D_plot)))

# ---------------- Plot heatmaps WITH annotated values ----------------
def _bin_edges_from_centers(centers):
    centers = np.asarray(centers, dtype=float)
    edges = np.empty(len(centers) + 1, dtype=float)
    if len(centers) > 1:
        edges[1:-1] = 0.5 * (centers[:-1] + centers[1:])
        edges[0]  = centers[0]  - 0.5 * (centers[1] - centers[0])
        edges[-1] = centers[-1] + 0.5 * (centers[-1] - centers[-2])
    else:
        edges[:] = [centers[0] - 0.5, centers[0] + 0.5]
    return edges

def _annotate_cells(ax, X_centers, Y_centers, M, fmt, vmin, vmax):
    threshold = 0.5 * (vmin + vmax)
    for iy, y in enumerate(Y_centers):
        for ix, x in enumerate(X_centers):
            val = M[iy, ix]
            color = "white" if val >= threshold else "black"
            ax.text(x, y, fmt.format(val), ha="center", va="center", color=color, fontsize=13)

def plot_pair_annotated(rho_a, rho_b, fname_base, which="pct"):
    """
    Make a 2-panel figure of annotated heatmaps.
    which = "pct" -> mean prevalence (%) with integer labels
            "D"   -> mean |D_s p(T)| in chosen units:
                      - if ANNOTATE_D_AS_PERCENT: percent with integer labels
                      - else: raw units with two decimals
    """
    fig, axes = plt.subplots(1, 2, figsize=(11.6, 4.8))

    # Percent axes for the sweep parameters
    X_centers = 100.0 * beta_grid  # columns
    Y_centers = 100.0 * tau_grid   # rows
    X_edges   = _bin_edges_from_centers(X_centers)
    Y_edges   = _bin_edges_from_centers(Y_centers)

    for ax, rho in zip(axes, [rho_a, rho_b]):
        df = all_results[rho]
        if which == "pct":
            vals = df.pivot(index="tau", columns="beta", values="E_pT_pct").loc[tau_grid, beta_grid].values
            vmin, vmax = vmin_pct, vmax_pct
            fmt = "{:.0f}"    # integer %
            cbar_label = "Mean $p(T)$ (%)"
            title = f"MB: mean prevalence (%) — $\\rho={rho:+.1f}$"
        else:
            vals = df.pivot(index="tau", columns="beta", values="E_absD_plot").loc[tau_grid, beta_grid].values
            vmin, vmax = vmin_Dp, vmax_Dp
            if ANNOTATE_D_AS_PERCENT:
                fmt = "{:.0f}"            # integer %
                cbar_label = r"Mean $|D_s p(T)|$ (%)"
                title = f"MB: mean $|D_s p(T)|$ (%) — $\\rho={rho:+.1f}$"
            else:
                fmt = "{:.2f}"            # two decimals in raw units
                cbar_label = r"Mean $|D_s p(T)|$"
                title = f"MB: mean $|D_s p(T)|$ — $\\rho={rho:+.1f}$"

        im = ax.imshow(
            vals, origin="lower", aspect="auto",
            extent=[X_edges[0], X_edges[-1], Y_edges[0], Y_edges[-1]],
            vmin=vmin, vmax=vmax
        )
        ax.set_title(title)
        ax.set_xlabel("Horizontal transmission $\\beta$ (%)")
        ax.set_ylabel("Vertical transmission $\\tau$ (%)")
        ax.set_xticks(X_centers)
        ax.set_yticks(Y_centers)
        ax.set_xlim(X_edges[0], X_edges[-1])
        ax.set_ylim(Y_edges[0], Y_edges[-1])

        # Colorbar
        divider = make_axes_locatable(ax)
        cax = divider.append_axes("right", size="4.5%", pad=0.10)
        cb = plt.colorbar(im, cax=cax)
        cb.set_label(cbar_label)

        # Annotate cells
        _annotate_cells(ax, X_centers, Y_centers, vals, fmt, vmin, vmax)

    fig.tight_layout()
    for ext in ("png", "pdf", "svg"):
        fig.savefig(fig_dir / f"{fname_base}.{ext}")
    plt.close(fig)

# Negative/zero and positive pairs — prevalence (%)
plot_pair_annotated(-0.5, 0.0, "mb_heatmaps_neg", which="pct")
plot_pair_annotated( 0.5, 0.9, "mb_heatmaps_pos", which="pct")

# Optional: the same for mean |D_s p(T)| in chosen units
plot_pair_annotated(-0.5, 0.0, "mb_heatmaps_neg_D", which="D")
plot_pair_annotated( 0.5, 0.9, "mb_heatmaps_pos_D", which="D")

print("Saved CSV grids to:", tbl_dir.as_posix())
print("Saved heatmap figures to:", fig_dir.as_posix())


Saved CSV grids to: heatMAPS/ihm_outputs_mb_sweeps/tables
Saved heatmap figures to: heatMAPS/ihm_outputs_mb_sweeps/figs
